In [ ]:
import math
import os
import sys
from typing import Optional

sys.path.append(os.path.abspath('.')) # to run files that are away
os.environ["WANDB_SILENT"] = "true"  # Suppress WandB logs

libraries = ["torch", "numpy", "polars"]
modules   = {lib: sys.modules.get(lib) for lib in libraries}

if not modules["torch"]:
    import torch
if not modules["numpy"]:
    import numpy as np
if not modules["polars"]:
    import polars as pl

import pandas as pd
import gc
import catboost as cb
import lightgbm as lgb
from pycatch22 import catch22_all
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split

from files_processor import LogFilesProcessor, WaferFilesProcessor, LogAndSpatialProcessor
from predictions import PrePredictionProcessor, SingleOutputModelPredictor, MultiOutputModelPredictor
from feature_selection import PCA_analysis, RFE_analysis
from asm_utils import Basics
from utils import Losses

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from autoencoder import Autoencoder, TrainAutoencoder

import asm_data_wrangling as asm
from key_params import NUM_WAFERS, step_col_name, COMMON_ID_COLS, COMMON_ID_COLS_MOD, parquet_folder_name#, dict_of_spatial_files, dict_of_log_files 

# device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

log_processor   = LogFilesProcessor(COMMON_ID_COLS_MOD, COMMON_ID_COLS, parquet_folder_name)
multi_predictor = MultiOutputModelPredictor(device)
single_predictor= SingleOutputModelPredictor(device)
preprocessor    = PrePredictionProcessor()


In [ ]:
"""Constants"""

main_folder   = "../ASM_data"
dict_of_spatial_files = {'file1': {'path': f"{main_folder}/2. marathon0/Wafer performance/Spatial property after step 4.csv", 'marathon': 0},
                         'file2': {'path': f"{main_folder}/3. marathon1/Wafer performance/Spatial property.csv", 'marathon': 1}}

dict_of_log_files = {'file1': {'path': f"{main_folder}/2. marathon0/logs/Step1.csv", 'step': 1, 'marathon': 0},
                     'file2': {'path': f"{main_folder}/2. marathon0/logs/Step2.csv", 'step': 2, 'marathon': 0},
                     'file3': {'path': f"{main_folder}/2. marathon0/logs/Step3.csv", 'step': 3, 'marathon': 0},
                     'file4': {'path': f"{main_folder}/2. marathon0/logs/Step4.csv", 'step': 4, 'marathon': 0},
                     'file5': {'path': f"{main_folder}/3. marathon1/logs/Step1.csv", 'step': 1, 'marathon': 1},
                     'file6': {'path': f"{main_folder}/3. marathon1/logs/Step2.csv", 'step': 2, 'marathon': 1},
                     'file7': {'path': f"{main_folder}/3. marathon1/logs/Step3.csv", 'step': 3, 'marathon': 1},
                     'file8': {'path': f"{main_folder}/3. marathon1/logs/Step4.csv", 'step': 4, 'marathon': 1},}
marathon_run_col = "marathon_run"
wafer_col        = "wafer"
run_col          = "#run"
step_id_col      = "step_id"
process_time_col = "process time"
radius_col       = "Radius (mm)"
site_id_col      = "Site #"
spatial_property_col = "Spatial property (nm)"


In [ ]:
"""code to split data into 4 wafers"""

# master_spatial_df, spatial_df_dict, y_df_dict, radius_wide_dict = asm.load_spatial_csv_and_create_targets(dict_of_spatial_files, main_folder, save=False)
# unique_marathon_runs_list = list(master_spatial_df["marathon_run"].unique())
# master_log_df = asm.load_and_process_and_combine_log_csv_files(dict_of_log_files, log_processor, unique_marathon_runs_list, step_col_name, main_folder, save=False)
# # master_log_df = master_log_df.fill_null(pl.lit(0))
# master_log_df = remove_constant_valued_cols(master_log_df)
# master_log_df = master_log_df.fill_null(pl.lit(0))

# # asm.dont_split_log_df_by_wafer_and_save_to_parquet(master_log_df, main_folder, overwrite = True)
# # asm.split_log_df_by_wafer_and_save_to_parquet(master_log_df, NUM_WAFERS, main_folder, log_processor, overwrite = True)

# # log_df_with_wafer_col = asm.infer_wafer_from_rc_values(master_log_df, rc_prefix="rc", wafer_col="wafer", overwrite = False)
# # log_df_with_wafer_col = asm.fast_infer_wafer(master_log_df, NUM_WAFERS)


In [ ]:
"""Code (new) to have a unified code/model (NOT split per wafer)"""

def generate_master_spatial_df(dict_of_spatial_files, main_folder):
    """Processing steps, starts from csv to a SPATIAL df"""
    master_spatial_df, unique_marathon_runs_list = LogAndSpatialProcessor.load_spatial_csv_files_to_1_df(dict_of_spatial_files)
    _, y_df = LogAndSpatialProcessor.create_target_df_from_spatial_df(master_spatial_df, main_folder, save=False)
    return master_spatial_df, unique_marathon_runs_list, y_df

def generate_master_log_df(unique_marathon_runs_list: list, keep_existing_runs: bool):
    """Processing steps, starts from csv to an exploded-by-wafer df log_df"""
    master_log_df = log_processor.load_and_process_and_combine_log_csv_files(dict_of_log_files, unique_marathon_runs_list, step_col_name, main_folder,
                                                                             keep_existing_runs, save_log_df_to_parquet = False)
    master_log_df = Basics.remove_constant_valued_cols(master_log_df)
    master_log_df = master_log_df.fill_null(pl.lit(0))
    master_log_df_exploded = LogAndSpatialProcessor.explode_log_df_rows_by_wafer(master_log_df, NUM_WAFERS)
    master_log_df_exploded = Basics.remove_constant_valued_cols(master_log_df_exploded)
    return master_log_df_exploded

_, unique_marathon_runs_list, y_df = generate_master_spatial_df(dict_of_spatial_files, main_folder)

parquet_folder      = "parquet_files"
log_df_parquet_file = "master_log_labelled.parquet"
parquet_file_path   = os.path.join(parquet_folder, log_df_parquet_file)

log_df_parquet_file2= "master_log_unlabelled.parquet"
parquet_file_path2  = os.path.join(parquet_folder, log_df_parquet_file2)

if not os.path.exists(parquet_file_path):
    os.makedirs(parquet_folder, exist_ok=True)

    # labelled
    master_log_df = generate_master_log_df(unique_marathon_runs_list, keep_existing_runs=True)
    master_log_df.write_parquet(parquet_file_path)

    # unlabelled
    master_log_df_unlabelled = generate_master_log_df(unique_marathon_runs_list, keep_existing_runs=False)
    master_log_df_unlabelled.write_parquet(parquet_file_path2)
    print("Saved parquet log files!")
else:
    master_log_df = pl.read_parquet(parquet_file_path)
    master_log_df_unlabelled = pl.read_parquet(parquet_file_path2)
    print("Loaded parquet log files!")


In [ ]:
# df_clean = master_log_df.select(master_log_df.columns[2:])
# mean_val = df_clean.select([pl.col("*").mean().mean()]).item()
# print("Mean of master_log_df:", mean_val)
mean_df = master_log_df.select([pl.col(col).mean().alias(col) for col in master_log_df.columns])
print(mean_df)

# df_clean = master_log_df_unlabelled.select(master_log_df_unlabelled.columns[2:])
# mean_val = df_clean.select(pl.mean_horizontal(pl.all())).item()
# print("Mean of master_log_df_unlabelled:", mean_val)
# mean_df2 = master_log_df_unlabelled.select([pl.col(col).mean().alias(col) for col in master_log_df_unlabelled.columns])
# print(mean_df2)

In [ ]:
"""Shaping + preprocessing"""

def _select_1_step_from_df(df: pl.DataFrame, step_number):
    return df.filter(pl.col(step_id_col) == step_number)

def reduce_df(reduction_method: str, df: pl.DataFrame, step_number: Optional[int] = None,
              N_downsampling: Optional[int] = None, N_last_rows_per_run: Optional[int] = None) -> pl.DataFrame:
    """df is too big, reduce it to speed things up
        - method: Reduction method, one of
            - "nothing": Do not reduce, return the original df
            - "subsample": Downsample rows (requires N_downsampling)
            - "subsample_1_step": Downsample single step
            - "latest_rows_per_run": Keep last N rows per (marathon_run, wafer) (requires N_last_rows_per_run)
            - "latest_rows_per_step_run": Keep last N rows per (step_id, marathon_run, wafer) (requires N_last_rows_per_run)
            - "1_step": Filter rows to a single step (requires step_number)
        - df: Input DataFrame to reduce.
        - step_number: Step number for "1_step" method.
        - N_downsampling: Downsampling factor for "subsample" method.
        - N_last_rows_per_run: Number of last rows to keep for "latest_rows_per_*" methods.
        - output: reduced polars df"""

    if reduction_method == "nothing":
        return df

    elif reduction_method == "subsample":
        subsampled_log_df = LogAndSpatialProcessor.downsample_df_rows(df, N_downsampling)
        return subsampled_log_df

    elif reduction_method == "subsample_1_step":
        if step_number is None or N_downsampling is None:
            raise ValueError("step_number and N_downsampling must be provided for 'subsample_1_step'")
        df_step = _select_1_step_from_df(df, step_number)
        return LogAndSpatialProcessor.downsample_df_rows(df_step, N_downsampling)

    elif reduction_method == "1_step":
        return _select_1_step_from_df(df, step_number)

    elif reduction_method == "latest_rows_per_run": # all runs
        latest_rows_per_run_df = (df
                                  .sort(process_time_col)
                                  .group_by([marathon_run_col, wafer_col])
                                  .tail(N_last_rows_per_run))
        return latest_rows_per_run_df

    elif reduction_method == "latest_rows_per_step_run": # all runs per step
        latest_rows_per_step_run_df = (df
                                       .sort(process_time_col)
                                       .group_by([step_id_col, marathon_run_col, wafer_col])
                                       .tail(N_last_rows_per_run))
        return latest_rows_per_step_run_df
    else:
        raise ValueError(f"Unknown reduction method: {reduction_method}")

reduction_method          = "subsample" # Options: nothing, subsample, subsample_1_step, latest_rows_per_run, latest_rows_per_step_run, 1_step
# labelled
reduced_log_df_labelled   = reduce_df(reduction_method, master_log_df, step_number=None, N_downsampling=10, N_last_rows_per_run=None)
log_df_labelled           = Basics.remove_constant_valued_cols(reduced_log_df_labelled)
y_df_expanded             = LogAndSpatialProcessor.expand_y_df_to_match_size_of_log_df(log_df_labelled, y_df)
log_df_labelled_no_str    = log_df_labelled.select(pl.exclude(pl.Utf8))

# unlabelled
reduced_log_df_unlabelled = reduce_df(reduction_method, master_log_df_unlabelled, step_number=None, N_downsampling=50, N_last_rows_per_run=None)
log_df_unlabelled         = Basics.remove_constant_valued_cols(reduced_log_df_unlabelled)
log_df_unlabelled_no_str  = log_df_unlabelled.select(pl.exclude(pl.Utf8))

#~~~~~~~~~~~~
# # split then scale X (avoids data leakage)
# X = log_df_labelled_no_str.to_pandas().drop(columns=[wafer_col, marathon_run_col, run_col], errors='ignore')
# y = y_df_expanded.drop(marathon_run_col, wafer_col).to_pandas()
# X_train, X_val, y_train, y_val  = train_test_split(X, y, test_size=0.2, random_state=42)
# X_train_scaled, X_val_scaled, _ = preprocessor.scale_X_after_split(X_train, X_val)
#~~~~~~~~~~~~

try:
    del master_log_df, master_log_df_unlabelled, reduced_log_df_labelled, reduced_log_df_unlabelled, log_df_unlabelled_no_str
        # log_df_labelled, y_df,\
        # subsampled_log_df, only_1_step_df, wide_radius_df, \
except NameError:
    pass

In [ ]:
"""[SotA] Make y multi-output predictor + target encode + use cat features into predictions"""

X = log_df_labelled.to_pandas()
X = X.drop(columns=[process_time_col])
y = y_df_expanded.drop([marathon_run_col, wafer_col]).to_pandas()

# turn 'marathon_run' into 'marathon'
X['marathon'] = X['marathon_run'].str.split('_', expand=True)[0].astype(int)
X[run_col]    = X[run_col].astype(str)
X = X.drop(columns = ['marathon_run'])

train_idx, val_idx = train_test_split(X.index, test_size=0.2, random_state=42)
X_train, y_train   = X.loc[train_idx], y.loc[train_idx]
X_val, y_val       = X.loc[val_idx],   y.loc[val_idx]

X_train, te_model  = Basics.apply_target_encoding_to_df(X_train, y_train, run_col, '#run_te')
X_val['#run_te']   = te_model.transform(X_val[run_col])

# Split into numeric and categorical
cat_features = ["#run_te", 'marathon']
# X_cat_train  = X_train[cat_features]
# X_cat_val    = X_val[cat_features]
# X_num_train  = X_train.drop(columns = cat_features)
# X_num_val    = X_val.drop(columns = cat_features)

# Scale only numeric, then combine with unscaled cat features
X_train_scaled, X_val_scaled, _ = preprocessor.scale_X_after_split(X_train, X_val, exclude_cols = cat_features)
# X_train_scaled = pd.concat([X_num_train_scaled, X_cat_train], axis=1).reset_index(drop=True)
# X_val_scaled   = pd.concat([X_num_val_scaled,   X_cat_val],   axis=1).reset_index(drop=True)
rmse_cat, y_pred_cat, _ = multi_predictor.predict_catboost(X_train_scaled, y_train.values, X_val_scaled,
                                                           y_val.values, cat_features=None)
print(f"CatBoost RMSE (multioutput+target encoding): {rmse_cat:.5f} nm")

# Combine train + val sets
X_full = pd.concat([X_train_scaled, X_val_scaled], axis=0).reset_index(drop=True)
y_full = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)
rmse_full, y_pred_full, importances = multi_predictor.predict_catboost2(X_full, y_full, X_val=None, y_val=None,
                                                                        cat_features=None, n_splits=3)
print(f"CatBoost CV RMSE (full set+target encoding): {rmse_full:.5f} nm")


In [ ]:
"""TSfresh Timeseries extraction. Options: wavelets, shapelets, deep models"""

from tsfresh import extract_features, select_features
from tsfresh.utilities.dataframe_functions import impute

X_L_pd                = log_df_labelled.to_pandas()
X_L_pd['combined_id'] = X_L_pd['marathon_run'].astype(str) + '_' + X_L_pd['wafer'].astype(str)

cols_to_keep = ['wafer', 'process time', 'combined_id'] + \
               [col for col in X_L_pd.columns if X_L_pd[col].dtype in ['float64', 'int64']]
df_numeric   = X_L_pd[cols_to_keep]

# 1. Extract features
X_features   = extract_features(df_numeric, column_id='combined_id', column_sort='process time')
print(X_features.shape)

# 2. Impute missing values
impute(X_features)
print("NaNs after impute:", X_features.isna().sum().sum())

# 3. Select relevant features using y
y_df_pd = y_df.to_pandas() if isinstance(y_df, pl.DataFrame) else y_df
y_df_pd['combined_id'] = y_df_pd['marathon_run'].astype(str) + '_' + y_df_pd['wafer'].astype(str)
y_df_pd = y_df_pd.set_index('combined_id').loc[X_features.index]
y_df_pd = y_df_pd.drop(columns=['wafer', 'marathon_run'])

# 4. Feature selection
def apply_tsfresh_feature_selection(X_features: pd.DataFrame, y_df_pd: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Selects relevant features using tsfresh.select_features, which Kendall's tau to select features relevant to each target (FDR>0.05)
        - X_features (pd.DataFrame): Feature matrix from TSFresh.
        - y_df_pd (pd.DataFrame): Target dataframe aligned by 'combined_id'.
        - tuple[pd.DataFrame, pd.DataFrame]: Selected features and aligned targets"""
    selected_cols = set()
    for target_col in y_df_pd.columns:
        y_target         = y_df_pd[target_col]
        selected_features= select_features(X_features, y_target)
        selected_cols.update(selected_features.columns)
    X_selected = X_features[list(selected_cols)]
    print(X_selected.shape)

    X_selected = X_selected.reset_index(drop=True)
    y          = y_df_pd.reset_index(drop=True)
    return X_selected, y
X_selected, y = apply_tsfresh_feature_selection(X_features, y_df_pd)

# 5. Train model
rmse, y_pred, importances, tsfresh_catboost_models = multi_predictor.predict_catboost2(X_selected, y, X_val=None, y_val=None,
                                                                                       cat_features=None, n_splits=1, return_final_model = False)
print(f"CatBoost CV RMSE (full set+target encoding): {rmse:.5f} nm")

Basics.save_catboost_models(tsfresh_catboost_models, "saved_models/tsfresh")


In [ ]:
"""Catch22. Much faster than tsfresh"""

def extract_catch22_features(df, id_cols, time_col):
    df               = df.copy()
    df['combined_id']= df[id_cols[0]].astype(str) + '_' + df[id_cols[1]].astype(str)
    numeric_cols     = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] and col not in id_cols + [time_col]]
    grouped          = df.groupby('combined_id')

    features_list = []
    index_list    = []

    for name, group in grouped:
        group_sorted = group.sort_values(time_col)
        feats = []
        for col in numeric_cols:
            ts = group_sorted[col].values
            feats.extend(catch22_all(ts)['values'])
        features_list.append(feats)
        index_list.append(name)

    col_names = [f"{col}_c22_{i}" for col in numeric_cols for i in range(22)]
    return pd.DataFrame(features_list, index=index_list, columns=col_names)

def align_y_targets_to_X_and_drop_index_cols(y_df, df, id_cols):
    """align X and y rows"""
    y_df = y_df.to_pandas() if isinstance(y_df, pl.DataFrame) else y_df
    y_df['combined_id'] = y_df[id_cols[0]].astype(str) + '_' + y_df[id_cols[1]].astype(str)
    y_df = y_df.set_index('combined_id').loc[df.index].reset_index()
    y_df = y_df.drop(columns=id_cols + ['combined_id', 'index'], errors='ignore')
    y_df = y_df.select_dtypes(include=['float64', 'int64'])
    return y_df

def get_top_feature_importance_for_each_target(df: pd.DataFrame, top_N_importances: int = 1) -> pd.DataFrame:
    """Return the top N most important features for each target_idx based on importance_mean
        - df: DataFrame with columns ['feature', 'target_idx', 'importance_mean']
        - top_n: Number of top features to select per target
        - DataFrame with top N features per target_idx sorted by importance_mean"""
    features_by_target_importance = (df.groupby('target_idx', group_keys=False)
                                    .apply(lambda x: x.nlargest(top_N_importances, 'importance_mean')))
    return features_by_target_importance

def get_intersection_of_2_df(df1: pd.DataFrame, df2: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return 2 df with columns aligned to their intersection"""
    common_cols = df1.columns.intersection(df2.columns)
    return df1[common_cols], df2[common_cols]


# labelled
X_L_pd      = log_df_labelled.to_pandas()
X_L_catch22 = extract_catch22_features(X_L_pd, id_cols=['marathon_run', 'wafer'], time_col='process time')
print(f"Labelled before optimiz.:  {int(X_L_catch22.memory_usage(deep=True).sum()/(1024*1024))} MB, shape: {X_L_catch22.shape}")
X_L_catch22 = Basics.optimize_df_memory(X_L_catch22, 1)
print(f"Labelled after optimiz.:  {int(X_L_catch22.memory_usage(deep=True).sum()/(1024*1024))} MB, shape: {X_L_catch22.shape}")
X_L_catch22 = Basics.remove_constant_valued_cols(X_L_catch22)
# X_L_catch22 = Basics.drop_and_impute_nan_cols(X_L_catch22, threshold=0.8, impute='zero')
y_df_pd     = align_y_targets_to_X_and_drop_index_cols(y_df, X_L_catch22, id_cols=['marathon_run', 'wafer'])

# unlabelled
X_U_pd      = log_df_unlabelled.to_pandas()
X_U_catch22 = extract_catch22_features(X_U_pd, id_cols=['marathon_run', 'wafer'], time_col='process time')
print(f"Unlabelled before optimiz.:  {int(X_U_catch22.memory_usage(deep=True).sum()/(1024*1024))} MB, shape: {X_U_catch22.shape}")
X_U_catch22 = Basics.optimize_df_memory(X_U_catch22, 1)
print(f"Unlabelled after optimiz.:  {int(X_U_catch22.memory_usage(deep=True).sum()/(1024*1024))} MB, shape: {X_U_catch22.shape}")
X_U_catch22 = Basics.remove_constant_valued_cols(X_U_catch22)
# X_U_catch22 = Basics.drop_and_impute_nan_cols(X_U_catch22, threshold=0.8, impute='zero')

# """Predict on X_label"""
# rmse, y_pred, importances, catch22_catboost_models = multi_predictor.predict_catboost2(X_L_catch22, y_df_pd, X_val=None, y_val=None,
#                                                                                        cat_features=None, n_splits=1, return_final_model = False)
# print(f"CatBoost CV RMSE (catch22 per-var): {rmse:.5f} nm")
# Basics.save_catboost_models(catch22_catboost_models, "saved_models/catch22")
# # features_by_target_importance = get_top_feature_importance_for_each_target(importances, 1)
# # features_by_target_importance.head(60)

"""catch22 model, unlabelled data"""

X_L_aligned, X_U_aligned = get_intersection_of_2_df(X_L_catch22, X_U_catch22)
print(f"aligned shape of X_L, X_U: {X_L_aligned.shape}, {X_U_aligned.shape}")

y_df_pd = align_y_targets_to_X_and_drop_index_cols(y_df, X_L_aligned, id_cols=['marathon_run', 'wafer'])

# SAVE CATCH22 MODELS
pseudolabel_model_path = "saved_models/catch22/pseudolabelling"
if os.path.exists(pseudolabel_model_path):
    print(f"Loading existing catch22 models...")
    catch22_catboost_models = Basics.load_all_catboost_models_in_dir(pseudolabel_model_path)
    y_pseudo = np.column_stack([model.predict(X_U_aligned) for model in catch22_catboost_models])
else:
    rmse, y_pred, _, catch22_catboost_models = multi_predictor.predict_catboost2(X_L_aligned, y_df_pd, X_val=None, y_val=None,
                                                                                 cat_features=None, n_splits=1, return_final_model = False)
    print(f"CatBoost CV RMSE (catch22 per-var): {rmse:.5f} nm")
    Basics.save_catboost_models(catch22_catboost_models, pseudolabel_model_path)


# NOTE: X_U_aligned is downsampled, the more we take out, the lower its mean ( no clue why)


In [ ]:
print("X_L_pd", X_L_pd.mean(numeric_only=True).mean())
print("X_U_pd", X_U_pd.mean(numeric_only=True).mean())

print("X_L_catch22", X_L_catch22.mean().mean())
print("X_U_catch22", X_U_catch22.mean().mean())

print("X_L_aligned", X_L_aligned.mean().mean())
print("X_U_aligned", X_U_aligned.mean().mean())

# X_U_pd.head(25)
print(X_L_pd.shape)
print(X_U_pd.shape)


In [ ]:
"""memory optimization"""

df = X_U_catch22

def print_mem(df, label):
    mem = df.memory_usage(deep=True).sum() / 1_048_576  # MB
    print(f"{label} memory: {mem:.2f} MB")

print_mem(X_L_pd, "Raw input")
X_L_catch22 = extract_catch22_features(X_L_pd, id_cols=['marathon_run', 'wafer'], time_col='process time')
print_mem(X_L_catch22, "After feature extraction")

X_features_opt = Basics.optimize_df_memory(X_L_catch22, nan_threshold=1)
print_mem(X_features_opt, "After optimization")


In [ ]:
"""ensemble model training, multimodel on each target"""

from sklearn.neighbors import NearestNeighbors
from catboost import CatBoostRegressor
from typing import List, Tuple, Optional
import pickle

class EnsemblePseudoLabeling:
    """Pseudo-labeling pipeline using CatBoost ensemble for multi-target regression"""

    def __init__(self, n_ensemble: int = 3, top_k: int = 50, keep_ratio: float = 0.8, cat_features=None,
                 model_path: str = None, pickle_dir: str = None, confidence_method: str = 'variance'):
        self.n_ensemble       = n_ensemble
        self.top_k            = top_k
        self.keep_ratio       = keep_ratio
        self.cat_features     = cat_features
        self.confidence_method= confidence_method
        self.model_path       = model_path or f"saved_models/catch22/ensemble_{n_ensemble}_keep{int(keep_ratio*100)}"
        self.pickle_dir       = pickle_dir or "pickled"
        self.ensemble_models  = []
        self.ensemble_model   = None

    def train_ensemble_models(self, X: pd.DataFrame, y: np.ndarray) -> list:
        """Train an ensemble of CatBoost models for each target in multi-target regression.
        For each target column in y, trains `n_ensemble` separate CatBoost models with randomized hyperparameters.
        Useful for ensembling and uncertainty estimation in multi-output regression tasks.
            - X (pd.DataFrame): Feature matrix.
            - y (np.ndarray): Target array of shape (n_samples, n_targets).
            - n_ensemble (int): Number of models to train per target.
            - cat_features (optional): List of categorical feature indices or names.
            - list: A list of length n_targets, where each element is a list of `n_ensemble` trained CatBoostRegressor models."""

        n_targets      = y.shape[1]
        seeds          = [42 + i for i in range(self.n_ensemble)]
        ensemble_models= []

        for target_idx in range(n_targets):
            models = []
            for seed in seeds:
                model = CatBoostRegressor(
                    iterations    = 50,
                    learning_rate = np.random.choice([0.1, 0.2, 0.3]),
                    depth         = np.random.choice([6, 7, 8]),
                    l2_leaf_reg   = np.random.choice([3, 5]),
                    bagging_temperature = np.random.uniform(0.5, 1.5),
                    task_type = 'CPU', verbose=0, random_seed=seed)
                model.fit(X, y[:, target_idx], cat_features=self.cat_features)
                models.append(model)
            ensemble_models.append(models)
        return ensemble_models

    def predict_ensemble(self, X_unlabeled: pd.DataFrame):# -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Predicts y_unlabeled using ensemble and returns predictions, variance, and uncertainty"""
        # shape: (n_targets, n_ensemble, n_samples)
        ensemble_preds     = np.array([[model.predict(X_unlabeled) for model in models]
                                      for models in self.ensemble_models])
        ensemble_var       = np.var(ensemble_preds, axis=1)   # shape: (n_targets, n_samples)
        sample_uncertainty = ensemble_var.mean(axis=0)        # shape: (n_samples,)
        return ensemble_preds, ensemble_var, sample_uncertainty

    def _retain_top_confident_predictions_by_neighbors(self, y_pseudo: np.ndarray, y_pred_labeled: np.ndarray,
                                                       X_unlabeled: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """Keep top X% most confident pseudo-labels based on distance to labeled predictions.
            y_pseudo: (n_unlabeled, n_targets) predicted pseudo-labels
            y_pred_labeled: (n_labeled, n_targets) predicted labels on labeled data
            X_unlabeled: (n_unlabeled, n_features) corresponding features
            keep_ratio: float in (0, 1), % of most confident samples to keep
            X_filtered: subset of X_unlabeled kept
            y_filtered: corresponding high-confidence pseudo-labels
            NOTE: set neighbors=1 to get distance to closest labeled prediction"""
        X_unlabeled = X_unlabeled.values if hasattr(X_unlabeled, 'values') else X_unlabeled
        neighbors   = NearestNeighbors(n_neighbors=1).fit(y_pred_labeled)
        dists, _    = neighbors.kneighbors(y_pseudo)
        dists       = dists.flatten()
        threshold   = np.quantile(dists, self.keep_ratio)
        keep_mask   = dists <= threshold
        return X_unlabeled[keep_mask], y_pseudo[keep_mask]

    def filter_confident_samples(self, X_unlabeled: pd.DataFrame, ensemble_preds: np.ndarray,
                                 uncertainty: np.ndarray, X_labeled: Optional[pd.DataFrame] = None,
                                 y_labeled_preds: Optional[np.ndarray] = None):# -> Tuple[pd.DataFrame, np.ndarray]:
        """Selects high-confidence samples from unlabeled data"""
        if self.confidence_method == 'variance':
            threshold   = np.quantile(uncertainty, self.keep_ratio)
            keep_mask   = uncertainty <= threshold
            X_confident = X_unlabeled[keep_mask]
            y_confident = ensemble_preds.mean(axis=1)[:, keep_mask].T  # shape: (n_kept, n_targets)
            return X_confident, y_confident
        elif self.confidence_method == 'distance':
            y_pseudo = ensemble_preds.mean(axis=1).T
            if y_labeled_preds is None:
                raise ValueError("y_labeled_preds required for distance-based confidence")
            X_conf_arr = X_unlabeled.values if hasattr(X_unlabeled, 'values') else X_unlabeled
            X_confident, y_confident = self._retain_top_confident_predictions_by_neighbors(y_pseudo, y_labeled_preds, X_conf_arr)
            return pd.DataFrame(X_confident, columns=X_unlabeled.columns), y_confident
        elif self.confidence_method == 'top_k':
            k           = min(self.top_k, len(uncertainty))
            sorted_idx  = np.argsort(uncertainty)
            keep_idx    = sorted_idx[:k]
            X_confident = X_unlabeled.iloc[keep_idx]
            y_confident = ensemble_preds.mean(axis=1)[:, keep_idx].T
            return X_confident, y_confident
        elif self.confidence_method == 'z_score':
            mean        = np.mean(uncertainty)
            std         = np.std(uncertainty)
            z_thresh    = -1 # keep samples below mean -1 stdev
            keep_mask   = uncertainty < mean + z_thresh * std
            X_confident = X_unlabeled[keep_mask]
            y_confident = ensemble_preds.mean(axis=1)[:, keep_mask].T
            return X_confident, y_confident
        else:
            raise ValueError(f"Unknown confidence_method: {self.confidence_method}")

    def retrain_model_on_combined_data(self, X_labeled: pd.DataFrame, y_labeled: np.ndarray, X_pseudo: pd.DataFrame,
                                       y_pseudo: np.ndarray, multi_predictor, overwrite: bool = False):# -> Optional[List[CatBoostRegressor]]:
        """Retrains CatBoost model using original + pseudo-labeled data. Optionally overwrite the model (ie when X shape changes)"""
        X_combined = pd.concat([X_labeled, X_pseudo], axis=0)
        y_combined = np.vstack([y_labeled, y_pseudo])

        # if os.path.exists(self.model_path) and (not overwrite):
        #     print("Loading existing models...")
        #     return Basics.load_all_catboost_models_in_dir(self.model_path)
        
        rmse, _, _, final_model = multi_predictor.predict_catboost2(X_combined, y_combined, cat_features=self.cat_features,
                                                                    n_splits=1, return_final_model=True)
        print(f"Retrained model RMSE on combined data: {rmse:.4f}")
        Basics.save_catboost_models(final_model, self.model_path)
        return final_model

    def run_ensemble_pseudolabel_pipeline(self, X_labeled: pd.DataFrame, y_labeled: np.ndarray, X_unlabeled: pd.DataFrame, 
                                          multi_predictor):# -> Tuple[Optional[List[CatBoostRegressor]], float]:
        """Main driver function for pseudo-labeling with ensemble models"""
        print("Step 1: Training ensemble of models per target")
        self.ensemble_models = self.train_ensemble_models(X_labeled, y_labeled)
        # with open(f"{self.pickle_dir}/ensemble_models.pkl", "wb") as f:
        #     pickle.dump(ensemble_models, f)

        print("Step 2: Predicting on unlabeled data using ensemble")
        ensemble_preds, ensemble_var, sample_uncertainty = self.predict_ensemble(X_unlabeled)
        # with open(f"{self.pickle_dir}/ensemble_preds.pkl", "wb") as f:
        #     pickle.dump((ensemble_preds, ensemble_var, sample_uncertainty), f)

        if self.confidence_method == 'distance':
            y_labeled_preds = np.column_stack([np.mean([model.predict(X_labeled) for model in target_models], axis=0)
                                               for target_models in self.ensemble_models])
        else:
            y_labeled_preds = None

        print("Step 3: Filtering high-confidence pseudo-labels")
        X_confident, y_confident = self.filter_confident_samples(X_unlabeled, ensemble_preds, sample_uncertainty,
                                                                 X_labeled=X_labeled,y_labeled_preds=y_labeled_preds)
        # X_confident.to_parquet(f"{self.pickle_dir}/X_confident.parquet")
        # np.save(f"{self.pickle_dir}/y_confident.npy", y_confident)

        print("Step 4: Retraining model on labeled + confident pseudo-labeled data")
        final_model = self.retrain_model_on_combined_data(X_labeled, y_labeled, X_confident, y_confident, multi_predictor)

        print("Step 5: Evaluating retrained model on original labeled data")
        y_pred = (np.column_stack([m.predict(X_labeled) for m in final_model])
                  if isinstance(final_model, list) else final_model.predict(X_labeled))
        rmse   = root_mean_squared_error(y_labeled, y_pred)
        print(f"RMSE on labeled set using retrained model: {rmse:.4f}")
        return final_model, rmse

    # ITERATIVE pipeline
    def run_iterative_pseudolabel_pipeline(self, X_labeled: pd.DataFrame, y_labeled: np.ndarray, X_unlabeled: pd.DataFrame,
                                           multi_predictor, X_val: pd.DataFrame, y_val: np.ndarray, n_iterations: int = 3):
        """Run pseudo-labeling pipeline iteratively to refine model with new pseudo-labeled data each round"""

        growth_increment = [0, 0.02, 0.02]
        prev_rmse        = np.inf

        for i in range(n_iterations):
            print(f"\n--- Iteration {i+1}/{n_iterations} ---")
            self.keep_ratio = min(0.5, self.keep_ratio + growth_increment[i])  # cap at 50%
            print("keep_ratio=",self.keep_ratio)

            # Step 1: Train ensemble on current labeled data
            self.ensemble_models = self.train_ensemble_models(X_labeled, y_labeled)

            # Step 1.5: get rmse as sanity check
            y_labeled_pred_ensemble = np.column_stack([
                np.mean([m.predict(X_labeled) for m in models], axis=0)
                for models in self.ensemble_models])
            rmse_iter = root_mean_squared_error(y_labeled, y_labeled_pred_ensemble)
            print(f"Iter. {i+1} RMSE on labeled data (ensemble): {rmse_iter:.4f}")

            # break if no improvement
            if prev_rmse - rmse_iter < 0.001:
                print(f"Early stopping: RMSE improvement too small ({prev_rmse:.4f} → {rmse_iter:.4f})")
                break
            prev_rmse = rmse_iter

            # Step 2: Predict unlabeled set
            ensemble_preds, _, uncertainty = self.predict_ensemble(X_unlabeled)
            
            # Step 3: Get distance-based support if needed
            if self.confidence_method == 'distance':
                y_labeled_preds = np.column_stack([np.mean([model.predict(X_labeled) for model in target_models], axis=0)
                                                  for target_models in self.ensemble_models])
            else:
                y_labeled_preds = None
            
            # Step 4: Select confident pseudo-labeled points
            X_pseudo, y_pseudo = self.filter_confident_samples(X_unlabeled, ensemble_preds, uncertainty, X_labeled, y_labeled_preds)
            
            if len(X_pseudo) == 0:
                print("No confident pseudo-labels found; stopping early.")
                break

            # Step 5: Append confident pseudo-labels to training set
            X_labeled = pd.concat([X_labeled, X_pseudo], axis=0)
            y_labeled = np.vstack([y_labeled, y_pseudo])
            print(f"Total samples after pseudo-labeling: {len(X_labeled)}")

            # Step 5 sanity check
            y_val_pred_iter = np.column_stack([np.mean([model.predict(X_val) for model in models], axis=0)
                                               for models in self.ensemble_models])
            rmse_val_iter   = root_mean_squared_error(y_val, y_val_pred_iter)
            print(f"Iter. {i+1} RMSE on valid. set: {rmse_val_iter:.4f}")

            if i == 0:
                prev_val_rmse = rmse_val_iter
            elif prev_val_rmse - rmse_val_iter < 0.001:
                print(f"Early stopping: Validation RMSE plateaued ({prev_val_rmse:.4f} → {rmse_val_iter:.4f})")
                break
            else:
                prev_val_rmse = rmse_val_iter

            # Step 5.5: remove selected pseudo-labeled samples from X_unlabeled
            keep_idx    = ~X_unlabeled.index.isin(X_pseudo.index)
            X_unlabeled = X_unlabeled[keep_idx]

            if len(X_unlabeled) == 0:
                print("No unlabeled data left; stopping.")
                break

        # Final model training
        final_model = self.retrain_model_on_combined_data(X_labeled, y_labeled, pd.DataFrame(), np.empty((0, y_labeled.shape[1])), multi_predictor)

        print("Step 5: Evaluating retrained model on original labeled data")
        y_pred = (np.column_stack([m.predict(X_labeled) for m in final_model])
                  if isinstance(final_model, list) else final_model.predict(X_labeled))
        rmse   = root_mean_squared_error(y_labeled, y_pred)
        print(f"RMSE on labeled set using retrained model: {rmse:.4f}")
        return final_model, rmse


# NOTE: train_ensemble_models() and retrain_model_on_combined_data() use DIFFERENT models, should be the same

In [ ]:
"""Evaluating ensemble pseudolabeling"""

pickle_dir     = "pickled"
conf_method    = "top_k" # options: variance, distance, top_k
# keep_ratio     = 0.03
# pseudo_labeler = EnsemblePseudoLabeling(keep_ratio=keep_ratio, n_ensemble=3, cat_features=None, pickle_dir=pickle_dir, confidence_method=conf_method)
# os.makedirs(pickle_dir, exist_ok=True)
# final_model, rmse = pseudo_labeler.run_iterative_pseudolabel_pipeline(X_labeled=X_L_aligned,y_labeled=y_df_pd.values,
#                                                                       X_unlabeled=X_U_aligned,multi_predictor=multi_predictor, n_iterations= 3)
# ===
# retrain_from_scratch = True
# if retrain_from_scratch:
    # print("retraining from scratch")
# final_model, rmse= pseudo_labeler.run_ensemble_pseudolabel_pipeline(X_labeled=X_L_aligned,y_labeled=y_df_pd.values,
#                                                                     X_unlabeled=X_U_aligned,multi_predictor=multi_predictor)
# else:
#     with open(f"{pickle_dir}/ensemble_models.pkl", "rb") as f:
#         ensemble_models = pickle.load(f)
#     with open(f"{pickle_dir}/ensemble_preds.pkl", "rb") as f:
#         ensemble_preds, ensemble_var, sample_uncertainty = pickle.load(f)

#     X_confident = pd.read_parquet(f"{pickle_dir}/X_confident.parquet")
#     y_confident = np.load(f"{pickle_dir}/y_confident.npy")

#     final_model = pseudo_labeler.retrain_model_on_combined_data(
#         X_labeled=X_L_aligned,
#         y_labeled=y_df_pd.values,
#         X_pseudo=X_confident,
#         y_pseudo=y_confident,
#         multi_predictor=multi_predictor)

#     y_pred = (np.column_stack([m.predict(X_L_aligned) for m in final_model]))
#     rmse   = root_mean_squared_error(y_df_pd.values, y_pred)
#     print(f"RMSE on labeled set using retrained model: {rmse:.4f}")

# ================================
# anti leakage

# train/val split BEFORE pseudo-labeling
X_train, X_val_sub, y_true_train, y_val_sub = train_test_split(X_L_aligned, y_df_pd.values, test_size=0.1, random_state=42)

# Run pseudo-labeling only on X_train
keep_ratio     = 0.01
pseudo_labeler = EnsemblePseudoLabeling(top_k = 50, keep_ratio=keep_ratio, n_ensemble=2, cat_features=None, pickle_dir=pickle_dir, confidence_method=conf_method)

final_model, rmse = pseudo_labeler.run_iterative_pseudolabel_pipeline(
    X_labeled   = X_train,
    y_labeled   = y_true_train,
    X_unlabeled = X_U_aligned,
    multi_predictor = multi_predictor,
    X_val = X_val_sub,
    y_val = y_val_sub,
    n_iterations = 2)

# Evaluate cleanly
y_pred_on_val = (np.column_stack([m.predict(X_val_sub) for m in final_model])
                 if isinstance(final_model, list)
                 else final_model.predict(X_val_sub))
rmse = root_mean_squared_error(y_val_sub, y_pred_on_val)
print(f"Final RMSE on held-out validation set: {rmse:.4f}")

rmse_baseline, _, _, _ = multi_predictor.predict_catboost2(X_train, y_true_train, X_val=X_val_sub,
                                                           y_val=y_val_sub, cat_features=None, n_splits=1)
print(f"Baseline RMSE on held-out validation: {rmse_baseline:.4f}")


In [ ]:
"""Mean teacher params"""
batch_size         = 256       # For both labeled and unlabeled loaders
dropout_ratio      = 0.15
ema_decay          = 0.9      # EMA decay factor for teacher model update
hidden_dims        = [64, 32]  # MLP architecture
lambda_u           = 0.18 # min(1.0, epoch / 10 * 0.5)      # Weight for consistency (unsupervised) loss
learning_rate      = 1e-3     # Initial learning rate
num_epochs         = 60       # Main training epochs
pseudo_conf_thresh = 1.5      # Confidence threshold for pseudo-label filtering
retrain_batch_size = 32       # Batch size for retraining on combined data
retrain_epochs     = 30       # Retraining epochs
retrain_lr         = 1e-4     # Learning rate for final retraining
student_noise      = 0.05
adam_weight_decay  = 1e-4


In [ ]:
"""Mean teacher"""

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class TabularRegressor(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, hidden_dims=hidden_dims):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            # layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_ratio))
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

X_L_aligned   = X_L_aligned.dropna(axis=1, how='all')
X_U_aligned = X_U_aligned.dropna(axis=1, how='all')
X_L_aligned   = X_L_aligned.loc[:, X_L_aligned.nunique() > 1]
X_U_aligned = X_U_aligned.loc[:, X_U_aligned.nunique() > 1]

# Impute remaining NaNs with column mean
X_labeled   = X_L_aligned.fillna(X_L_aligned.mean())
X_unlabeled = X_U_aligned.fillna(X_U_aligned.mean())

# Scale + tensor
scaler      = StandardScaler().fit(np.vstack([X_labeled.values, X_unlabeled.values]))
X_labeled   = torch.tensor(scaler.transform(X_labeled), dtype=torch.float32).to(device)
X_unlabeled = torch.tensor(scaler.transform(X_unlabeled), dtype=torch.float32).to(device)
y_labeled   = torch.tensor(y_df_pd.values, dtype=torch.float32).to(device)

labeled_loader   = DataLoader(TensorDataset(X_labeled, y_labeled), batch_size=batch_size, shuffle=True)
unlabeled_loader = DataLoader(TensorDataset(X_unlabeled), batch_size=batch_size, shuffle=True)

student = TabularRegressor(X_labeled.shape[1], y_labeled.shape[1]).to(device)
teacher = TabularRegressor(X_labeled.shape[1], y_labeled.shape[1]).to(device)
teacher.load_state_dict(student.state_dict())

optimizer = torch.optim.AdamW(student.parameters(), lr=learning_rate, weight_decay=adam_weight_decay)

for epoch in range(num_epochs):
    lambda_u_epoch = lambda_u * min(1.0, epoch / 5)  # linear ramp over 5 epochs

    student.train(); teacher.eval()
    total_loss     = 0
    unlabeled_iter = iter(unlabeled_loader)
    for labeled_x, labeled_y in labeled_loader:
        try:
            (unlabeled_x,) = next(unlabeled_iter)
        except StopIteration:
            unlabeled_iter = iter(unlabeled_loader)
            (unlabeled_x,) = next(unlabeled_iter)

        # Supervised loss
        pred_l   = student(labeled_x)
        loss_sup = F.mse_loss(pred_l, labeled_y)

        # Unsupervised consistency loss
        with torch.no_grad():
            teacher_preds = teacher(unlabeled_x)
        # student_preds = student(unlabeled_x)
        noisy_x       = unlabeled_x + student_noise * torch.randn_like(unlabeled_x)
        student_preds = student(noisy_x)
        loss_unsup    = F.mse_loss(student_preds, teacher_preds)

        # Total loss
        # loss = loss_sup + lambda_u * loss_unsup
        loss = loss_sup + lambda_u_epoch * loss_unsup

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # EMA update for teacher
        with torch.no_grad():
            for t, s in zip(teacher.parameters(), student.parameters()):
                t.data.mul_(ema_decay).add_(s.data * (1 - ema_decay))
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f"Epoch {epoch+1} loss: {total_loss/len(labeled_loader):.4f}")


def evaluate_rmse(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = model(X).cpu().numpy()
    rmse = root_mean_squared_error(y.cpu().numpy(), preds)
    return rmse

def pseudo_labeling(teacher, X_unlabeled, threshold=0.1):
    teacher.eval()
    with torch.no_grad():
        preds = teacher(X_unlabeled)
    preds_np = preds.cpu().numpy()
    # Use confidence = inverse of prediction std dev as example (or use your own metric)
    conf = 1 / (preds_np.std(axis=1) + 1e-6)
    high_conf_mask = conf > threshold
    return X_unlabeled[high_conf_mask], preds.detach()[high_conf_mask]

rmse = evaluate_rmse(teacher, X_labeled, y_labeled)
print(f"RMSE on labeled data: {rmse:.4f}")

X_pseudo, y_pseudo = pseudo_labeling(teacher, X_unlabeled, threshold=pseudo_conf_thresh)
print(f"Pseudo-labeled samples: {len(X_pseudo)}")

# Add pseudo-labeled to labeled for next training round:
X_labeled = torch.cat([X_labeled, X_pseudo], dim=0)
y_labeled = torch.cat([y_labeled, y_pseudo], dim=0)

# Combine labeled and pseudo-labeled data into one dataset
combined_X = X_labeled 
combined_y = y_labeled

combined_loader = DataLoader(TensorDataset(combined_X, combined_y), batch_size=retrain_batch_size, shuffle=True)

# Retrain student model on combined data
student   = TabularRegressor(combined_X.shape[1], combined_y.shape[1]).to(device)
optimizer = torch.optim.AdamW(student.parameters(), lr=retrain_lr)

for epoch in range(retrain_epochs):
    student.train()
    total_loss = 0
    for batch_x, batch_y in combined_loader:
        optimizer.zero_grad()
        preds = student(batch_x)
        loss  = F.mse_loss(preds, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f"Retrain Epoch {epoch+1} loss: {total_loss/len(combined_loader):.4f}")

# Evaluate retrained model
rmse = evaluate_rmse(student, X_labeled, y_labeled)
print(f"RMSE after retraining: {rmse:.4f}")

# NOTE: need to replace model with catboost model

In [ ]:
"""[Supervised] autoencoder components"""

this_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device from module: {this_device}")

dropout_prob=        0.05
ae_training_epochs=  100        # training epochs
ae_batch_size=       126        # number of samples per batch
ae_optimizer_lr=     0.0001#0.00011     # learning rate for the optimizer
weight_decay=        0.00001    # for L2 regularization
training_patience=   80         # how many epochs with no improvement to stop training
scheduler_patience=  80
scheduler_mode=      'min'      # min (max) reduces learning rate when validation loss stops improving (starts increasing)
scheduler_factor=    0.8        # multiplies lr by this factor when validation loss plateaus

X_L_aligned, X_U_aligned = Basics.drop_shared_high_nan_cols(X_L_aligned, X_U_aligned, 0.5)

# Fill NaNs with mean
X_L_aligned.fillna(X_L_aligned.mean(), inplace=True)
X_U_aligned.fillna(X_U_aligned.mean(), inplace=True)
assert X_L_aligned.columns.equals(X_U_aligned.columns)

X_L_tensor = torch.tensor(X_L_aligned.values, dtype=torch.float32)
X_U_tensor = torch.tensor(X_U_aligned.values, dtype=torch.float32)
y_tensor   = torch.tensor(y_df_pd.values, dtype=torch.float32)

print("X_L_tensor mean:", X_L_tensor.mean().item())
print("X_U_tensor mean:", X_U_tensor.mean().item())

scaler = StandardScaler()
scaler.fit(X_L_tensor.numpy())
X_L_scaled = torch.tensor(scaler.transform(X_L_tensor.numpy()), dtype=torch.float32)
X_U_scaled = torch.tensor(scaler.transform(X_U_tensor.numpy()), dtype=torch.float32)

print("X_L_scaled mean:", X_L_scaled.mean().item())
print("X_U_scaled mean:", X_U_scaled.mean().item())


# Use full labeled set (no val split)
train_dataset      = TensorDataset(X_L_scaled, y_tensor)
unlabelled_dataset = TensorDataset(X_U_scaled, torch.empty(len(X_U_scaled)))
train_loader       = DataLoader(train_dataset, batch_size=ae_batch_size, shuffle=True)
unlabelled_loader  = DataLoader(unlabelled_dataset, batch_size=ae_batch_size, shuffle=True)


In [ ]:

print(X_U_scaled.min().item(), X_U_scaled.max().item(), X_U_scaled.mean().item(), X_U_scaled.std().item())


In [ ]:
input_size     = X_L_scaled.shape[1]
# layer_dims     = [input_size, 1024, 256, 64, 16] # 0.111
# layer_dims     = [input_size, 1200, 300, 80, 20, 5]
layer_dims     = [input_size, 256, 64, 16]
projection_dim = 5
pred_hidden_dim= 32 # hidden dim for prediction head
mode           = "reconstruct" # options: ["reconstruct", "projection", "latent", "predict"]
autoencoder    = Autoencoder(layer_dims, dropout_prob=dropout_prob, projection_dim=projection_dim,
                             mode=mode, pred_dim=y_tensor.shape[1], pred_hidden_dim=pred_hidden_dim)
optimizer      = torch.optim.AdamW(autoencoder.parameters(), lr=ae_optimizer_lr, weight_decay=weight_decay)
scheduler      = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, scheduler_mode, patience=scheduler_patience, factor=scheduler_factor)

# Train or load AE
should_we_save_ae = True
autoencoder_path  = 'autoencoder.pth'
if not os.path.exists(autoencoder_path) or should_we_save_ae:
    print("Training new autoencoder...")
    trainer   = TrainAutoencoder()
    best_loss = trainer.train_autoencoder(this_device, autoencoder, ae_training_epochs, train_loader, optimizer, scheduler,
                                          validation_loader = None, patience = training_patience)
    print(f"best loss: {best_loss:.3f}")

    # save trained AE
    torch.save(autoencoder.state_dict(), autoencoder_path)
else:
    print("Loading existing autoencoder model...")
    autoencoder = Autoencoder(layer_dims, dropout_prob=dropout_prob, projection_dim=projection_dim,
                              mode=mode, pred_dim=y_tensor.shape[1])
    autoencoder.load_state_dict(torch.load(autoencoder_path, map_location=this_device))
    autoencoder.to(this_device)

# running AE
autoencoder.eval()
with torch.no_grad():
    for X_batch, _ in train_loader:
        X_batch = X_batch.to(this_device)
        X_hat   = autoencoder(X_batch, mode="reconstruct")
        # Z, H = autoencoder(X_batch, mode="latent")
        # H = autoencoder(X_batch, mode="projection")


In [ ]:
"""Full Loss Pipeline"""

def augment_tensor_with_noise(X: torch.Tensor, noise_std: float = 0.1) -> torch.Tensor:
    """augment tensor. Options: add Gaussian noise, dropout, mixup, shuffling"""
    noise = torch.randn_like(X) * noise_std
    return X + noise

def train_step(autoencoder: nn.Module, optimizer: torch.optim.Optimizer, X_L: torch.Tensor, y_L: torch.Tensor,
               X_U: torch.Tensor, temperature: float, device: torch.device, w_recon: float = 0.2, w_contrast: float = 0.6, w_pred: float = 0.3) -> float:
    """Performs a single training step for the autoencoder with multi-loss objectives.
    - autoencoder (nn.Module): Autoencoder model with support for 'reconstruct', 'projection', 'predict' modes.
    - optimizer (torch.optim.Optimizer): Optimizer for model parameters.
    - X_L (torch.Tensor): Labelled input data.
    - y_L (torch.Tensor): Supervised targets for X_L.
    - X_U (torch.Tensor): Unlabelled input data.
    - temperature (float): Temperature scaling for contrastive loss.
    - w_recon (float): Weight for reconstruction loss.
    - w_contrast (float): Weight for contrastive loss.
    - w_pred (float): Weight for supervised prediction loss.
    - output (float): Total loss value for the training step."""
    autoencoder.train()
    optimizer.zero_grad()

    X_L = X_L.to(device)
    X_U = X_U.to(device)
    y_L = y_L.to(device)

    # Augment inputs
    X_L_aug = augment_tensor_with_noise(X_L, 0.0)
    X_U_aug = augment_tensor_with_noise(X_U, 0.0)

    # Forward passes
    X_L_reconstr = autoencoder(X_L, mode="reconstruct")
    X_U_reconstr = autoencoder(X_U, mode="reconstruct")
    y_pred       = autoencoder(X_L, mode="predict") # but what is the prediction method here?

    # latents
    Z_U      = autoencoder.forward(X_U, mode="latent")

    # projections (for contrastive loss)
    H_L      = autoencoder(X_L, mode="projection")
    H_U      = autoencoder(X_U, mode="projection")
    H_L_aug  = autoencoder(X_L_aug, mode="projection")
    H_U_aug  = autoencoder(X_U_aug, mode="projection")

    # losses
    loss_reconstr = Losses.compute_MSE_loss(X_L, X_L_reconstr) + Losses.compute_MSE_loss(X_U, X_U_reconstr)
    loss_contrast = Losses.get_contrastive_loss(H_L, H_L_aug, temperature) + \
                    Losses.get_contrastive_loss(H_U, H_U_aug, temperature)
    loss_predict  = Losses.compute_MSE_loss(y_L, y_pred)

    # print("X_L_reconstr", X_L_reconstr)
    # print(f"Recon Loss: {loss_reconstr:.4f}, Contrast Loss: {loss_contrast:.4f}, Predict Loss: {loss_predict:.4f}")
    total_loss = w_recon * loss_reconstr + w_contrast * loss_contrast + w_pred * loss_predict
    total_loss.backward()
    optimizer.step()



    # Inside train_step, after forward passes
    print(f"X_L_input stats (min/max/mean/std): {X_L.min().item():.2f}/{X_L.max().item():.2f}/{X_L.mean().item():.2f}/{X_L.std().item():.2f}")
    print(f"X_L_reconstr stats (min/max/mean/std): {X_L_reconstr.min().item():.2f}/{X_L_reconstr.max().item():.2f}/{X_L_reconstr.mean().item():.2f}/{X_L_reconstr.std().item():.2f}")
    print(f"X_U_input stats (min/max/mean/std): {X_U.min().item():.2f}/{X_U.max().item():.2f}/{X_U.mean().item():.2f}/{X_U.std().item():.2f}")
    print(f"X_U_reconstr stats (min/max/mean/std): {X_U_reconstr.min().item():.2f}/{X_U_reconstr.max().item():.2f}/{X_U_reconstr.mean().item():.2f}/{X_U_reconstr.std().item():.2f}")
    print(f"y_L_target stats (min/max/mean/std): {y_L.min().item():.2f}/{y_L.max().item():.2f}/{y_L.mean().item():.2f}/{y_L.std().item():.2f}")
    print(f"y_pred stats (min/max/mean/std): {y_pred.min().item():.2f}/{y_pred.max().item():.2f}/{y_pred.mean().item():.2f}/{y_pred.std().item():.2f}")
    return total_loss.item()

# Training loop
w_recon    = 0.8
w_contrast = 0.1
w_pred     = 0.1

unlabelled_iter = iter(unlabelled_loader)
for epoch in range(ae_training_epochs):
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        # You need a batch of unlabelled data too; can cycle unlabelled_loader or sample randomly
        # For example:
        # X_U_batch, _ = next(iter(unlabelled_loader))
        try:
            X_U_batch, _ = next(unlabelled_iter)
        except StopIteration:
            unlabelled_iter = iter(unlabelled_loader)
            X_U_batch, _ = next(unlabelled_iter)

        loss = train_step(autoencoder= autoencoder, optimizer=optimizer, X_L=X_batch, y_L=y_batch, X_U=X_U_batch, temperature=0.5,
                          w_recon=w_recon, w_contrast=w_contrast, w_pred=w_pred, device=device)
        epoch_loss += loss

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Average Loss: {epoch_loss / len(train_loader):.4f}")



In [ ]:
# get train_loader dataloader min, max, mean, std
for X_batch, _ in train_loader:
    print(X_batch.min(), X_batch.max(), X_batch.mean(), X_batch.std()) 

print(X_L_scaled.mean(), X_L_scaled.std())



In [ ]:
"""Dimensionality"""

from utils import DimensionalityEstimator
dimensionality = DimensionalityEstimator.estimate_dataset_dimensionality(X_train_scaled.numpy(), 10)
print(dimensionality)


def get_real_dimensionality_from_PCA(X: pd.DataFrame, variance: float = 0.95) -> int:
    """Estimate intrinsic dimensionality of the dataset using PCA"""
    X   = X.dropna(axis=1, how='all')
    X   = X.fillna(0)
    pca = PCA()
    pca.fit(X)
    cumulative_variance = pca.explained_variance_ratio_.cumsum()
    intrinsic_dim = (cumulative_variance < variance).sum() + 1
    print(f"Intrinsic dimensionality ({variance*100}% variance): {intrinsic_dim}")
    return intrinsic_dim

get_real_dimensionality_from_PCA(X_train_scaled.numpy(), 0.99)
get_real_dimensionality_from_PCA(X_train_scaled.numpy(), 0.99)


In [ ]:
print(torch.isnan(X_all).any())
# X_all = torch.nan_to_num(X_all, nan=0.0)

# Count columns where all values are NaN
fully_nan_cols = X_L_aligned.isna().all()
num_fully_nan  = fully_nan_cols.sum()
print(f"{num_fully_nan} columns are fully NaN")

# Optionally, list them:
print("Fully NaN columns:", fully_nan_cols[fully_nan_cols].index.tolist())

nan_percent = X_L_aligned.isna().mean() * 100
print(nan_percent[nan_percent > 50].sort_values(ascending=False))


In [ ]:
"""FixMatch"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Model ---
class TabularRegressor(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, hidden_dims=[128, 64]):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

# --- Augmentations ---
def weak_augment(x: torch.Tensor, noise_std=0.001) -> torch.Tensor:
    return x
    # out = x + torch.randn_like(x) * noise_std
    # assert not torch.isnan(out).any(), "NaN in weak augment"
    # return out

def strong_augment(x: torch.Tensor, noise_std=0.005, drop_prob=0.2) -> torch.Tensor:
    return x
    # noise = torch.randn_like(x) * noise_std
    # mask = (torch.rand_like(x) > drop_prob).float()
    # out = (x + noise) * mask
    # assert not torch.isnan(out).any(), "NaN in strong augment"
    # return out

# --- FixMatch loss ---
def fixmatch_loss(model, labeled_x, labeled_y, unlabeled_x, threshold=0.3):
    preds_l = model(labeled_x)
    loss_sup = F.mse_loss(preds_l, labeled_y)

    weak   = weak_augment(unlabeled_x)
    strong = strong_augment(unlabeled_x)

    with torch.no_grad():
        pseudo_labels = model(weak)
        confidence = pseudo_labels.abs().mean(dim=1)  # shape [batch_size]
        mask = confidence > threshold

    if mask.any():
        preds_strong = model(strong[mask])
        loss_unsup = F.mse_loss(preds_strong, pseudo_labels[mask])
    else:
        loss_unsup = torch.tensor(0.0, device=unlabeled_x.device)

    return loss_sup + loss_unsup, loss_sup.item(), loss_unsup.item()

# --- Clean data ---
# Drop cols with all NaN
X_L_aligned = X_L_aligned.dropna(axis=1, how='all')
X_U_aligned = X_U_aligned.dropna(axis=1, how='all')

# Drop constant-valued cols BEFORE imputation
X_L_aligned = Basics.remove_constant_valued_cols(X_L_aligned)
X_U_aligned = Basics.remove_constant_valued_cols(X_U_aligned)

# Impute remaining NaNs with mean
X_labeled = X_L_aligned.fillna(X_L_aligned.mean())
X_unlabeled = X_U_aligned.fillna(X_U_aligned.mean())

# Check no NaNs remain
assert X_labeled.isna().sum().sum() == 0
assert X_unlabeled.isna().sum().sum() == 0

# --- Scale ---
all_features = np.vstack([X_labeled.values, X_unlabeled.values])
scaler = StandardScaler().fit(all_features)

X_labeled_scaled = scaler.transform(X_labeled.values)
X_unlabeled_scaled = scaler.transform(X_unlabeled.values)

y_scaler = StandardScaler()
y_scaled = y_scaler.fit_transform(y_df_pd.values)

# --- Tensor conversion ---
X_labeled = torch.tensor(X_labeled_scaled, dtype=torch.float32).to(device)
y_labeled = torch.tensor(y_df_pd.values, dtype=torch.float32).to(device)
X_unlabeled = torch.tensor(X_unlabeled_scaled, dtype=torch.float32).to(device)

assert not torch.isnan(X_labeled).any()
assert not torch.isnan(X_unlabeled).any()
assert not torch.isnan(y_labeled).any()
assert not torch.isinf(X_labeled).any()
assert not torch.isinf(X_unlabeled).any()
assert not torch.isinf(y_labeled).any()

# --- Model and optimizer ---
model = TabularRegressor(input_dim=X_labeled.shape[1], output_dim=y_labeled.shape[1]).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

batch_size = 32
labeled_loader = DataLoader(TensorDataset(X_labeled, y_labeled), batch_size=batch_size, shuffle=True)
unlabeled_loader = DataLoader(TensorDataset(X_unlabeled), batch_size=batch_size, shuffle=True)

# --- Training loop ---
for epoch in range(20):
    model.train()
    unlabeled_iter = iter(unlabeled_loader)
    total_loss = total_sup = total_unsup = 0

    for labeled_x, labeled_y in labeled_loader:
        try:
            (unlabeled_x,) = next(unlabeled_iter)
        except StopIteration:
            unlabeled_iter = iter(unlabeled_loader)
            (unlabeled_x,) = next(unlabeled_iter)

        optimizer.zero_grad()
        loss, loss_sup_val, loss_unsup_val = fixmatch_loss(model, labeled_x, labeled_y, unlabeled_x)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_sup  += loss_sup_val
        total_unsup += loss_unsup_val

    print(f"Epoch {epoch+1} | Total: {total_loss:.1f} | Sup: {total_sup:.1f} | Unsup: {total_unsup:.1f}")

In [ ]:
"""[repeat] training on pseudolabels"""

from sklearn.neighbors import NearestNeighbors

def retain_top_confident_predictions_by_neighbors(y_pseudo: np.ndarray, y_pred_labeled: np.ndarray, X_unlabeled: np.ndarray,
                                                  keep_ratio: float) -> tuple[np.ndarray, np.ndarray]:
    """Keep top X% most confident pseudo-labels based on distance to labeled predictions.
        y_pseudo: (n_unlabeled, n_targets) predicted pseudo-labels
        y_pred_labeled: (n_labeled, n_targets) predicted labels on labeled data
        X_unlabeled: (n_unlabeled, n_features) corresponding features
        keep_ratio: float in (0, 1), % of most confident samples to keep
        X_filtered: subset of X_unlabeled kept
        y_filtered: corresponding high-confidence pseudo-labels
        NOTE: set neighbors=1 to get distance to closest labeled prediction"""
    nbrs      = NearestNeighbors(n_neighbors=1).fit(y_pred_labeled)
    dists, _  = nbrs.kneighbors(y_pseudo)
    dists     = dists.flatten()
    threshold = np.quantile(dists, keep_ratio)
    keep_mask = dists <= threshold
    return X_unlabeled[keep_mask], y_pseudo[keep_mask]

def retain_top_confident_by_variance(y_pseudo_ensemble: np.ndarray, X_unlabeled: np.ndarray, keep_ratio: float):
    """Keep top X% least uncertain pseudo-labels based on variance across ensemble.
    y_pseudo_ensemble: (n_models, n_samples, n_targets)
    X_unlabeled: (n_samples, n_features)
    keep_ratio: float in (0,1)"""
    variances = np.var(y_pseudo_ensemble, axis=0).mean(axis=1)  # shape: (n_samples,)
    threshold = np.quantile(variances, keep_ratio)
    keep_mask = variances <= threshold
    return X_unlabeled[keep_mask], y_pseudo_ensemble[:, keep_mask, :].mean(axis=0)

# remove if not used
def iterative_pseudo_labeling(model, X_labeled: np.ndarray, y_labeled: np.ndarray, X_unlabeled: np.ndarray,
                              keep_ratio: float, max_iter: int = 10) -> tuple[np.ndarray, np.ndarray]:
    """Iteratively pseudo-label unlabeled data using a confidence-based filter.
        model: regresssion model with fit/predict interface
        X_labeled: initial labeled features
        y_labeled: initial labeled targets
        X_unlabeled: features to pseudo-label
        keep_ratio: fraction of most confident samples to keep per iteration
        max_iter: maximum iterations to prevent infinite loop
        X_final: full training data (labeled + pseudo-labeled)
        y_final: corresponding targets"""

    for _ in range(max_iter):
        model.fit(X_labeled, y_labeled)
        y_pseudo = model.predict(X_unlabeled)
        y_pred_labeled = model.predict(X_labeled)

        # Select high-confidence pseudo-labels
        X_conf, y_conf = retain_top_confident_predictions_by_neighbors(y_pseudo, y_pred_labeled, X_unlabeled, keep_ratio)

        if len(X_conf) == 0:
            break  # nothing confident left

        # Add confident pseudo-labeled samples to labeled set
        X_labeled = np.concatenate([X_labeled, X_conf], axis=0)
        y_labeled = np.concatenate([y_labeled, y_conf], axis=0)

        # Remove used samples from unlabeled pool
        mask = np.ones(len(X_unlabeled), dtype=bool)
        idx_conf = np.isin(X_unlabeled, X_conf).all(axis=1)
        mask[idx_conf] = False
        X_unlabeled = X_unlabeled[mask]

        if len(X_unlabeled) == 0:
            break  # all samples labeled
    return X_labeled, y_labeled

keep_ratio = 0.5 # higher = more data used (less confident)
X_unlabeled_confident, y_pseudo_confident = retain_top_confident_predictions_by_neighbors(y_pseudo, y_pred, X_U_aligned, keep_ratio)
# y_pseudo_ensemble = np.array([model.predict(X_U_aligned) for model in catboost_models])
# X_unlabeled_confident, y_pseudo_confident = retain_top_confident_by_variance(y_pseudo_ensemble, X_U_aligned.values, keep_ratio)

X_combined = np.concatenate([X_L_aligned, X_unlabeled_confident], axis=0)
X_combined = pd.DataFrame(X_combined, columns=X_L_aligned.columns)
y_combined = np.concatenate([y_df_pd.values, y_pseudo_confident], axis=0)

print('about to predict')
partial_label_model_path = "saved_models/catch22/partial_label_model"
if os.path.exists(partial_label_model_path):
    combined_model = Basics.load_all_catboost_models_in_dir(partial_label_model_path)
else:
    rmse, y_pred_combined, importances, partial_label_model = multi_predictor.predict_catboost2(X_combined, y_combined, X_val=None, y_val=None,
                                                                                                cat_features=None, n_splits=1, return_final_model=True)
    Basics.save_catboost_models(partial_label_model, partial_label_model_path)
print('done')

# Convert DataFrame to numpy arrays if needed
X_unlabeled_confident_arr = X_unlabeled_confident.values if hasattr(X_unlabeled_confident, 'values') else X_unlabeled_confident
X_features_unlabelled_arr = X_U_aligned.values if hasattr(X_U_aligned, 'values') else X_U_aligned

set_confident = set(map(tuple, X_unlabeled_confident_arr))
mask_used = np.array([tuple(row) in set_confident for row in X_features_unlabelled_arr])

# Now mask_used length == X_features_unlabelled_arr.shape[0]
X_remaining = X_U_aligned.loc[~mask_used] if hasattr(X_U_aligned, 'loc') else X_features_unlabelled_arr[~mask_used]

if hasattr(partial_label_model, 'estimators_'):  # MultiOutputRegressor case
    y_pseudo_remaining = np.column_stack([est.predict(X_remaining) for est in partial_label_model.estimators_])
else:  # list of models case
    y_pseudo_remaining = np.column_stack([model.predict(X_remaining) for model in partial_label_model])


# 2. Train new model on combined (pseudo + real)
print('about to predict2')
combined_model_path = "saved_models/catch22/combined_model"
if os.path.exists(combined_model_path):
    combined_model = Basics.load_all_catboost_models_in_dir(combined_model_path)
else:
    rmse_train, y_pred2, _, combined_model = multi_predictor.predict_catboost2(X_combined, y_combined, cat_features=None,
                                                                               n_splits=1, return_final_model=True)
    Basics.save_catboost_models(combined_model, combined_model_path)

# 3. Predict on labeled set
y_pred_on_labeled = (np.column_stack([m.predict(X_L_aligned) for m in combined_model])
                     if isinstance(combined_model, list)
                     else combined_model.predict(X_L_aligned))
# 4. Evaluate RMSE
score = rmse(y_df_pd.values, y_pred_on_labeled)
print(f"RMSE on labeled data using model trained on labeled + pseudo-labelled data: {score:.4f}")


In [ ]:
# get results on test set

from sklearn.model_selection import train_test_split

X_train_sub, X_val_sub, y_train_sub, y_val_sub = train_test_split(
    X_L_aligned, y_df_pd.values, test_size=0.2, random_state=42)

y_pred_on_labeled = (np.column_stack([m.predict(X_val_sub) for m in combined_model])
                     if isinstance(combined_model, list)
                     else combined_model.predict(X_val_sub))
score = root_mean_squared_error(y_val_sub, y_pred_on_labeled)
print(f"RMSE on labeled data using model trained on labeled + pseudo-labelled data: {score:.4f}")


In [ ]:
"""Plot y_pseudo (after 1round) + y_pseudo2 (after 2round)"""

plt.figure(figsize=(14, 8))

# Unlabelled predictions (blue)
for i in range(y_pseudo_remaining.shape[0]):
    if i% 10==0:
        plt.plot(y_pseudo_remaining[i], linestyle='-', color='red', linewidth=0.3, alpha=0.6)
        plt.plot(y_pseudo[i], linestyle='-', color='green', linewidth=0.3, alpha=0.6)
plt.title('Predictions')
plt.xlabel('site_id')
plt.ylabel('Spatial property (nm)')
plt.legend()
plt.show()


In [ ]:
"""Plot y_pred + y_pseudo"""

plt.figure(figsize=(14, 8))

# Unlabelled predictions (blue)
for i in range(y_pseudo.shape[0]):
    if i % 12 == 0:
        plt.plot(y_pseudo[i], linestyle='-', color='blue', linewidth=0.5, alpha=0.6)
plt.plot([], linestyle='-', color='blue', label='y_pred (no label)')

# Labelled predictions (red)
for row in y_pred:
    plt.plot(row, linestyle='-', color='red', linewidth=0.8, alpha=0.3)
plt.plot([], linestyle='-', color='red', label='y_pred (labelled)')

# From ASM (green)
for _, row in y_df_pd.iterrows():
    plt.plot(row.values, linestyle='-', color='green', linewidth=0.7, alpha=0.3)
plt.plot([], linestyle='-', color='green', label='y (from ASM)')

plt.title('Predictions')
plt.xlabel('site_id')
plt.ylabel('Spatial property (nm)')
plt.legend()
plt.show()


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
import umap
from adjustText import adjust_text

# Get shared columns + subset both + concat
common_cols = X_clean.columns.intersection(X_clean2.columns)
X_clean     = X_clean[common_cols]
X_clean2    = X_clean2[common_cols]
X_combined  = pd.concat([X_clean, X_clean2], axis=0)
X_combined  = X_combined.reindex(sorted(X_combined.columns), axis=1)  # Ensure col order

# Scale
scaler   = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_combined), columns=X_combined.columns, index=X_combined.index)

# Cluster all
kmeans = MiniBatchKMeans(n_clusters=18, batch_size=256, random_state=42)
cluster_labels = kmeans.fit_predict(X_scaled)

# UMAP
umap_reducer = umap.UMAP(random_state=42)
X_2d = umap_reducer.fit_transform(X_scaled)

# Build label mask (to color labeled/unlabeled separately)
is_labeled = np.zeros(len(X_combined), dtype=bool)
is_labeled[:len(X_clean)] = True

# Plot
plt.figure(figsize=(14,9))
plt.scatter(X_2d[~is_labeled, 0], X_2d[~is_labeled, 1], c=cluster_labels[~is_labeled], cmap='tab10', s=30, label='Unlabeled')
plt.scatter(X_2d[is_labeled, 0],  X_2d[is_labeled, 1],  c=cluster_labels[is_labeled],  cmap='tab10', s=3, label='Labeled')

texts = []
for i, txt in enumerate(X_scaled.index):
    if i % 9 == 0:
        texts.append(plt.text(X_2d[i, 0], X_2d[i, 1], str(txt), fontsize=9, alpha=0.8))
adjust_text(texts, arrowprops=dict(arrowstyle='-', color='black', lw=0.3))

plt.title("UMAP projection: labeled vs. unlabeled")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.colorbar(label='Cluster')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

# X_2d: 2D UMAP projection of X_features_labeled
# y_df_pd: your (labeled) DataFrame of 100 target columns

n_bins = 5
n_targets = 4  # number of targets to visualize

for col in y_df_pd.columns[:n_targets]:
    # Discretize the continuous target into bins
    binned = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile') \
             .fit_transform(y_df_pd[[col]]).flatten()

    # Plot UMAP with colors based on binned target values
    plt.figure(figsize=(6, 4))
    plt.scatter(X_2d[:, 0], X_2d[:, 1], c=binned, cmap='tab10', s=10)
    plt.title(f"UMAP projection colored by binned {col}")
    plt.axis('off')
    plt.colorbar(label='Binned class')
    plt.tight_layout()
    plt.show()


In [ ]:
for col in y_df_pd.columns[80:86]:  # show first 5 targets
    full_labels = np.full(len(X_clean), np.nan)
    full_labels[:len(y_df_pd)] = y_df_pd[col].values

    plt.figure(figsize=(6,4))
    plt.scatter(X_2d[:, 0], X_2d[:, 1], c=full_labels, cmap='plasma', s=8)
    plt.title(f"UMAP colored by y{col}")
    plt.axis('off')
    plt.colorbar()
    plt.tight_layout()
    plt.show()


In [ ]:
from sklearn.manifold import TSNE

X_2d = TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=42).fit_transform(X_scaled)

plt.figure(figsize=(8,6))
plt.scatter(X_2d[:,0], X_2d[:,1], c=cluster_labels, cmap='tab10', s=10)
plt.title("t-SNE projection of Catch22 features")
plt.xlabel("tSNE-1")
plt.ylabel("tSNE-2")
plt.colorbar(label='Cluster')
plt.tight_layout()
plt.show()


In [ ]:
"""[outdated] Explode log_df and predict single-output y. Obsolete, keeping the 100 y values in 1 row (for each wafer) does much better"""

# Ensure consistent types in both log and spatial data
log_df = log_df_labelled.with_columns([pl.col(wafer_col).cast(pl.Utf8),
                                                    pl.col(marathon_run_col).cast(pl.Utf8),
                                                    pl.col(run_col).cast(pl.Utf8)])
y_df = master_spatial_df.with_columns([
    pl.col(wafer_col).cast(pl.Utf8),
    pl.col(marathon_run_col).cast(pl.Utf8)])

y_subset    = y_df.select([marathon_run_col, wafer_col, site_id_col, radius_col, spatial_property_col])
exploded_df = log_df.join(y_subset, on=[marathon_run_col, wafer_col], how="left")

# Prepare y for regression: single target
y_expanded = exploded_df.select([
    marathon_run_col, wafer_col, site_id_col, spatial_property_col]).rename({spatial_property_col: "target"})

y_full = y_expanded.select("target").to_pandas()
X_full = exploded_df.drop([process_time_col, spatial_property_col, ]).with_columns([
    pl.col(wafer_col).cast(pl.Utf8),
    pl.col(site_id_col).cast(pl.Utf8),
    pl.col(run_col).cast(pl.Utf8),
    pl.col(marathon_run_col).cast(pl.Utf8)]).to_pandas()

# X_full=X_full.drop(columns=['Site #'])

# Extract 'marathon' from 'marathon_run'
X_full['marathon'] = X_full[marathon_run_col].str.split('_', expand=True)[0].astype(str)
X_full             = X_full.drop(columns = ['marathon_run'])

X_full, te_model = Basics.apply_target_encoding_to_df(X_full, y_full, run_col, '#run_te')
X_full, te_model = Basics.apply_target_encoding_to_df(X_full, y_full, "Site #", 'site#_te')

X_train, X_val, y_train, y_val = train_test_split(X_full, y_full, test_size=0.2, random_state=42)

cat_features = ['marathon', wafer_col, '#run_te', 'site#_te']#site_id_col,]
X_cat_train  = X_train[cat_features]
X_cat_val    = X_val[cat_features]

X_num_train  = X_train.drop(columns=cat_features)
X_num_val    = X_val.drop(columns=cat_features)

# Scale only numerics
X_num_train_scaled, X_num_val_scaled, _ = preprocessor.scale_X_after_split(X_num_train, X_num_val)

# Combine
X_train_scaled = pd.concat([X_num_train_scaled, X_cat_train], axis=1).reset_index(drop=True)
X_val_scaled   = pd.concat([X_num_val_scaled,   X_cat_val],   axis=1).reset_index(drop=True)

# Predict
rmse, y_pred, importances = single_predictor.predict_catboost_single_model(
    X_train_scaled, y_train, X_val_scaled, y_val, cat_features=None)
print(f"CatBoost RMSE (single target + encoding): {rmse:.5f} nm")


# with all 0.627
# without radius 0.625
# without (radius + siteID) 1.907 nm
# without siteID: 1.219
# with siteID encoded: 0.6314

In [ ]:
"""PCA + kmeans"""

# ===== PCA =====
pca_object = PCA_analysis()
pca_model  = pca_object.fit_pca(X_train_scaled[numeric_feature_names], var_threshold=0.99)

# Explained variance info
components, n_components = pca_object.explain_pca_variance(pca_model, show_plot=False)

# Transformed data already reduced to n_components
X_train_pca_reduced = pca_model.transform(X_train_scaled[numeric_feature_names])
X_val_pca_reduced   = pca_model.transform(X_val_scaled[numeric_feature_names])

rmse_pca, y_pred_pca, _ = single_predictor.predict_catboost_single_model(
    X_train_pca_reduced, y_train, X_val_pca_reduced, y_val, cat_features=None)
print(f"CatBoost RMSE (PCA reduced): {rmse_pca:.3f} nm")

# ===== SelectKBest =====
k_features = len(numeric_feature_names)
selector   = SelectKBest(score_func=f_regression, k = k_features)

X_train_selected = selector.fit_transform(X_train_num, y_train.values.ravel())
X_val_kbest      = selector.transform(X_val_scaled[numeric_feature_names])

selected_cols = X_train_num.columns[selector.get_support()]
# print(f"Selected features (SelectKBest): {selected_cols.tolist()}")

rmse_kbest, y_pred_kbest, _ = single_predictor.predict_catboost_single_model(
    X_train_selected, y_train, X_val_kbest, y_val, cat_features=None)
print(f"CatBoost RMSE (SelectKBest): {rmse_kbest:.3f} nm")

# ===== Kbest + PCA =======
# # Step 1: Select K best features
# k_features = len(numeric_feature_names)
# selector   = SelectKBest(score_func=f_regression, k = k_features)

# X_train_kbest = selector.fit_transform(X_train_scaled[numeric_feature_names], y_train.values.ravel())
# X_val_kbest   = selector.transform(X_val_scaled[numeric_feature_names])

# # Step 2: PCA on K-best features using your PCA_analysis class
# pca_object = PCA_analysis()
# pca_model  = pca_object.fit_pca(pd.DataFrame(X_train_kbest), var_threshold=0.99)

# X_train_kbest_pca = pca_model.transform(X_train_kbest)
# X_val_kbest_pca   = pca_model.transform(X_val_kbest)

# _, N_pca_components = pca_object.explain_pca_variance(pca_model, show_plot=False)

# # Step 3: CatBoost prediction
# rmse_kbest_pca, y_pred_kpca, _ = single_predictor.predict_catboost_single_model(
#     X_train_kbest_pca, y_train, X_val_kbest_pca, y_val, cat_features=None)
# print(f"CatBoost RMSE (KBest + PCA): {rmse_kbest_pca:.3f} nm")


In [ ]:
"""New feature eng. methods"""

from catboost import CatBoostRegressor
# from sklearn.feature_selection import SelectFromModel

# === Feature selection using CatBoost (includes categoricals) ===
catboost_model = CatBoostRegressor(verbose=0, random_state=42)
catboost_model.fit(X_train_scaled, y_train, cat_features = cat_features)

# Get feature importances
importances      = catboost_model.get_feature_importance()
feature_names    = X_train_scaled.columns
thres_percent    = 25 #17 # keep top (100-thres_percent)% features
selected_mask    = importances >= np.percentile(importances, thres_percent)

selected_columns = feature_names[selected_mask]
X_train_reduced  = X_train_scaled[selected_columns]
X_val_reduced    = X_val_scaled[selected_columns]

# Recompute categorical feature indices post-selection
cat_features_reduced = [i for i, col in enumerate(selected_columns) if col in cat_features]

# Predict with reduced feature set
rmse_sfm, _, _ = single_predictor.predict_catboost_single_model(
    X_train_reduced, y_train, X_val_reduced, y_val, cat_features=cat_features_reduced)
print(f"CatBoost RMSE (Selected Features): {rmse_sfm:.5f} nm")


sys.exit()

# Get sorted features by importance
support_mask    = selector.get_support()
importances     = selector.estimator_.feature_importances_
selected_cols   = X_train_num.columns[support_mask]
sorted_indices  = importances[support_mask].argsort()[::-1]
sorted_features = selected_cols[sorted_indices]
print(sorted_features)

sys.exit()

# ====== recursive feature elimination (RFE) with catboost ==========
X_train_num         = X_train_scaled.select_dtypes(include=np.number)
RFE_object          = RFE_analysis(device)
rmse_rfe, rfe_model = RFE_object.apply_recursive_feature_elimination(X_train_scaled, X_val_scaled, y_train, y_val, single_predictor, fraction_cols_to_keep = 0.99)
sorted_features     = RFE_object.get_sorted_features_by_importance(rfe_model, X_train_num)


# cached_rfe = {"selected_features": sorted_features.tolist(),
#               "importances": importances.tolist(),
#               "sorted_features": sorted_features.tolist(),
#               "estimator": rfe_model}
# import joblib
# joblib.dump(cached_rfe, "rfe_pass1_results.pkl")
# cached_rfe = joblib.load("rfe_pass1_results.pkl")


# pass #2
X_train_reduced     = X_train_scaled[sorted_features]
X_val_reduced       = X_val_scaled[sorted_features]
X_train_num_reduced = X_train_reduced.select_dtypes(include=np.number)
rmse_rfe, rfe_model = RFE_object.apply_recursive_feature_elimination(X_train_reduced, X_val_reduced, y_train,
                                                                     y_val, single_predictor, fraction_cols_to_keep=0.5)
sorted_features2    = RFE_object.get_sorted_features_by_importance(rfe_model, X_train_num_reduced)


# # ======== Drop Low-Importance Features (CatBoost)====
# model = CatBoostRegressor(verbose=0, random_state=42)
# model.fit(X_train_scaled, y_train, cat_features=cat_features)

# importances = model.get_feature_importance()
# keep_mask   = importances > np.percentile(importances, 25)

# X_train_imp = X_train_scaled.iloc[:, keep_mask]
# X_val_imp   = X_val_scaled.iloc[:, keep_mask]

# rmse_imp, _, _ = single_predictor.predict_catboost_single_model(
#     X_train_imp, y_train, X_val_imp, y_val, cat_features=None)
# print(f"CatBoost RMSE (Important features): {rmse_imp:.3f} nm")

# # ====== UMAP ========
# import umap
# X_train_num = X_train_scaled[numeric_feature_names]
# X_val_num   = X_val_scaled[numeric_feature_names]

# umap_model = umap.UMAP(n_components=10, random_state=42)
# X_train_umap = umap_model.fit_transform(X_train_num)
# X_val_umap   = umap_model.transform(X_val_num)

# rmse_umap, _, _ = single_predictor.predict_catboost_single_model(
#     X_train_umap, y_train, X_val_umap, y_val, cat_features=None)
# print(f"CatBoost RMSE (UMAP): {rmse_umap:.3f} nm")

# # ======= Polynomial Features + KBest =======
# from sklearn.preprocessing import PolynomialFeatures

# X_train_num = X_train_scaled[numeric_feature_names]
# X_val_num   = X_val_scaled[numeric_feature_names]

# poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
# X_train_poly = poly.fit_transform(X_train_num)
# X_val_poly   = poly.transform(X_val_num)

# selector = SelectKBest(score_func=f_regression, k=50)
# X_train_poly_sel = selector.fit_transform(X_train_poly, y_train.values.ravel())
# X_val_poly_sel   = selector.transform(X_val_poly)

# rmse_poly, _, _ = single_predictor.predict_catboost_single_model(
#     X_train_poly_sel, y_train, X_val_poly_sel, y_val, cat_features=None)
# print(f"CatBoost RMSE (Poly + SelectKBest): {rmse_poly:.3f} nm")


In [ ]:
print(sorted_features)

plt.figure(figsize=(16, 14))
for col in sorted_features:
    if (col not in log_df_labelled.columns) or (col.startswith("rc")):
        continue
    s = log_df_labelled[col]
    plt.plot(s, label=col, linewidth=2)
plt.legend(fontsize=10)
plt.show()

plt.figure(figsize=(16, 14))
for col in sorted_features:
    if (col not in log_df_labelled.columns):
        continue
    if (col.startswith("rc")):
        s = log_df_labelled[col]
        plt.plot(s, label=col, linewidth=2)
plt.legend(fontsize=10)
plt.show()


In [ ]:
"""Autoencoder"""

this_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device from module: {this_device}")

layer1_dim=          128         # layer 1 nodes
layer2_dim=          64          # layer 2 nodes
latent_dim=          32          # num of features in latent layer, get this from data dimensionality
dropout_prob=        0.05
ae_training_epochs=  100        # training epochs
ae_batch_size=       126        # number of samples per batch
ae_optimizer_lr=     0.0011     # learning rate for the optimizer
weight_decay=        0.00001    # for L2 regularization
training_patience=   80         # how many epochs with no improvement to stop training
scheduler_patience=  80
scheduler_mode=      'min'      # min (max) reduces elarning rate when validation loss stops improving (starts increasing)
scheduler_factor=    0.8        # multiplies lr by this factor when validation loss plateaus

reduced_log_df_numeric = reduced_log_df_labelled.select(pl.col(pl.NUMERIC_DTYPES))

X_scaled_tensor = torch.tensor(X_scaled.values, dtype=torch.float32)
X_train_torch   = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
X_val_torch     = torch.tensor(X_val.to_numpy(),   dtype=torch.float32)

train_dataset = TensorDataset(X_train_torch, torch.zeros(len(X_train)))  # dummy labels
val_dataset   = TensorDataset(X_val_torch, torch.zeros(len(X_val)))

# Create DataLoader objects for train and validation datasets
input_size   = X_train_torch.shape[1]
train_loader = DataLoader(train_dataset, batch_size=ae_batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=ae_batch_size, shuffle=False)

autoencoder  = Autoencoder(input_size, layer1_dim, layer2_dim, latent_dim, dropout_prob)
optimizer    = torch.optim.AdamW(autoencoder.parameters(), lr=ae_optimizer_lr, weight_decay=weight_decay)
scheduler    = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, scheduler_mode, patience=scheduler_patience, factor=scheduler_factor)

if not os.path.exists('autoencoder.pth'):
    trainer   = TrainAutoencoder()
    best_loss = trainer.train_autoencoder(this_device, autoencoder, ae_training_epochs, train_loader, optimizer, scheduler,
                                          validation_loader=val_loader, patience = training_patience)
    print(f"best loss: {best_loss:.3f}")

    # save trained AE
    torch.save(autoencoder.state_dict(), 'autoencoder.pth')


In [ ]:
"""Latents"""

from sklearn.decomposition import PCA

# load trained AE

autoencoder = Autoencoder(input_size, layer1_dim, layer2_dim, latent_dim, dropout_prob)
autoencoder.load_state_dict(torch.load('autoencoder.pth'))
autoencoder.to(this_device)
autoencoder.eval()

# Get latent space for X_train
with torch.no_grad():
    X_latent: np.ndarray       = autoencoder.encoder(X_scaled_tensor.to(this_device)).cpu().numpy()
    X_train_latent: np.ndarray = autoencoder.encoder(X_train_torch.to(this_device)).cpu().numpy()
    X_val_latent: np.ndarray   = autoencoder.encoder(X_val_torch.to(this_device)).cpu().numpy()
    
# 2D
plt.scatter(X_latent[:, 0], X_latent[:, 1], s=3, alpha=0.5)
plt.title("2D Latent Space")
plt.xlabel("Latent dim 1")
plt.ylabel("Latent dim 2")
plt.show()

# 3D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_latent)

plt.scatter(X_pca[:, 0], X_pca[:, 1], s=3, alpha=0.5)
plt.title("Latent Space (PCA to 2D)")
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.show()


In [ ]:
"""Decoder + PCA"""

# X_train_latent[:, 8] = 0
X_train_recon = autoencoder.decoder(torch.from_numpy(X_train_latent).to(this_device)).detach().cpu().numpy()
X_val_recon   = autoencoder.decoder(torch.from_numpy(X_val_latent).to(this_device)).detach().cpu().numpy()
X_recon       = autoencoder.decoder(torch.from_numpy(X_latent).to(this_device)).detach().cpu().numpy()
# ===

latent_df = pd.DataFrame(X_scaled, columns=[f"z{i}" for i in range(X_scaled.shape[1])])
latent_dim = X_scaled.shape[1]
# latent_df = pd.DataFrame(X_latent, columns=[f"z{i}" for i in range(X_latent.shape[1])])
target_df = pd.DataFrame(y_scaled, columns=[f"y{i}" for i in range(y_scaled.shape[1])])

# corr_matrix = latent_df.corr()
# sns.heatmap(corr_matrix.abs(), cmap='viridis')

corr_matrix = latent_df.corrwith(target_df, axis=0)# This won't work directly because corrwith compares series by index, not cross-columns.

# Instead, compute pairwise correlations manually:
corr_matrix = pd.DataFrame(
    np.corrcoef(latent_df.values.T, target_df.values.T)[:latent_dim, latent_dim:],
    index=latent_df.columns,
    columns=target_df.columns)

sns.heatmap(corr_matrix.abs(), cmap='viridis')
plt.xlabel('Targets')
plt.ylabel('Latent Features')
plt.show()


In [ ]:
"""Correlation of X with y"""

if not isinstance(X_scaled, pd.DataFrame):
    X_df = pd.DataFrame(X_scaled, columns=[f"x{i}" for i in range(X_scaled.shape[1])])
else:
    X_df = X_scaled.copy()

target_df = pd.DataFrame(y_scaled, columns=[f"y{i}" for i in range(y_scaled.shape[1])])

# Compute correlation matrix
corr_matrix = pd.DataFrame(
    np.corrcoef(X_df.values.astype(np.float64).T, target_df.values.astype(np.float64).T)[:X_df.shape[1], X_df.shape[1]:],
    index=X_df.columns,
    columns=target_df.columns)

# Get top input features by max absolute correlation across targets
top_N_features  = 50
avg_corr_per_feature = corr_matrix.abs().mean(axis=1)
top_features    = avg_corr_per_feature.nlargest(top_N_features).index
corr_matrix_top = corr_matrix.loc[top_features]

# Plot heatmap
plt.figure(figsize=(8, 10))
sns.heatmap(corr_matrix_top.abs(), cmap='coolwarm')
plt.xlabel('Targets')
plt.ylabel(f'Top {top_N_features} Input Features')
plt.yticks(ticks=np.arange(corr_matrix_top.shape[0]), labels=corr_matrix_top.index, fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
"""Kmeans clustering. NOTE make sure to apply it to the entire X dataset, not just runs which have a wafer"""

from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=2, random_state=42)
cluster_labels = kmeans.fit_predict(X_latent)

# # ===================
# plot 1
# X_pca = PCA(n_components=2).fit_transform(X_latent)
# plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='Set1', s=5)
# plt.title("Clustering in Latent Space")
# plt.show()

# # ===================
# plot 1.2
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, perplexity=10, max_iter=1500)
X_tsne = tsne.fit_transform(X_latent)

plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=cluster_labels, cmap='Set1', s=5)
plt.title("Clustering in Latent Space (t-SNE)")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.show()

# # ===================
# # plot 1.2
# import umap.umap_ as umap  # if you installed umap-learn

# # Fit UMAP to reduce latent space to 2D
# reducer = umap.UMAP(n_components=2, random_state=42)
# X_umap = reducer.fit_transform(X_latent)

# # KMeans clustering (same as before)
# kmeans = KMeans(n_clusters=12, random_state=42)
# cluster_labels = kmeans.fit_predict(X_latent)

# # Plot UMAP projection with cluster colors
# plt.scatter(X_umap[:, 0], X_umap[:, 1], c=cluster_labels, cmap='Set1', s=5)
# plt.title("Clustering in Latent Space (UMAP)")
# plt.xlabel("UMAP 1")
# plt.ylabel("UMAP 2")
# plt.show()


In [ ]:
"""[PCA] High-variance features are NOT always predictive, use PCA components as a safe bet.
PCA components = linear combinations of features, capture global structure, NOT local feature importance
NOTE that PCA does not look at y, only X"""

pca_obj   = PCA_analysis()
pca_model = pca_obj.fit_pca(X_scaled)
X_pca     = pca_model.transform(X_scaled)

top_N_pca_components, N_pca_components = pca_obj.explain_pca_variance(pca_model, var_threshold=0.99)
X_pca_reduced = X_pca[:, :N_pca_components]

pca_obj.print_top_features_per_component(X_scaled, top_N_pca_components, top_k_features=2)
sorted_features = pca_obj.summarize_feature_importance(X_scaled, top_N_pca_components, top_k_features=5)


In [ ]:
"""SimCLR"""

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Dataset with augmentations (simple: random noise)
class SimCLRDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def augment(self, x):
        # Dropout (feature masking)
        mask_prob = 0.05
        mask = (torch.rand_like(x) > mask_prob).float()
        x_masked = x * mask

        # Jitter (feature scaling)
        scale = 0.9 + 0.2 * torch.rand_like(x)
        x_scaled = x_masked * scale

        # Noise (gaussian noise)
        noise = 0.05 * torch.randn_like(x_masked)
        return x_scaled + noise

    def __getitem__(self, idx):
        x = self.X[idx]
        return self.augment(x), self.augment(x)

# Simple MLP encoder for tabular data
class Encoder(nn.Module):
    def __init__(self, input_dim, latent_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim),
            nn.ReLU(),
        )
    def forward(self, x):
        return self.net(x)

# Projection head as in SimCLR paper
class ProjectionHead(nn.Module):
    def __init__(self, latent_dim=128, proj_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, proj_dim)
        )
    def forward(self, x):
        return self.net(x)

# NT-Xent loss for SimCLR
def nt_xent_loss(z1, z2, temperature=0.5):
    batch_size = z1.shape[0]
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    representations = torch.cat([z1, z2], dim=0)
    similarity_matrix = torch.matmul(representations, representations.T)

    # Mask to ignore similarity with self
    mask = (~torch.eye(2*batch_size, 2*batch_size, dtype=bool)).to(z1.device)

    positives = torch.cat([torch.diag(similarity_matrix, batch_size),
                           torch.diag(similarity_matrix, -batch_size)], dim=0)

    negatives = similarity_matrix[mask].view(2*batch_size, -1)

    logits = torch.cat([positives.unsqueeze(1), negatives], dim=1)
    labels = torch.zeros(2*batch_size, dtype=torch.long).to(z1.device)

    logits = logits / temperature
    loss = F.cross_entropy(logits, labels)
    return loss

# Usage example
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_scaled_tensor = torch.tensor(X_scaled.values, dtype=torch.float32)
y_scaled_tensor = torch.tensor(y_scaled, dtype=torch.float32)

dataset = SimCLRDataset(X_scaled_tensor)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

input_dim = X_scaled_tensor.shape[1]
encoder = Encoder(input_dim).to(device)
proj_head = ProjectionHead().to(device)
optimizer = torch.optim.AdamW(list(encoder.parameters()) + list(proj_head.parameters()), lr=0.0015)

for epoch in range(10):
    total_loss = 0
    for x1, x2 in dataloader:
        x1, x2 = x1.to(device), x2.to(device)
        h1 = encoder(x1)
        h2 = encoder(x2)
        z1 = proj_head(h1)
        z2 = proj_head(h2)
        loss = nt_xent_loss(z1, z2)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")


In [ ]:
"""t-SNE"""

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

encoder.eval()
with torch.no_grad():
    embeddings = encoder(torch.tensor(X_scaled.values, dtype=torch.float32).to(device)).cpu().numpy()

tsne = TSNE(n_components=2, random_state=42)
emb_2d = tsne.fit_transform(embeddings)

plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=y_scaled_tensor.argmax(dim=1).cpu().numpy(), cmap='tab10', s=5)
plt.title('TSNE of SimCLR Embeddings')
plt.show()


In [ ]:

# rmse_cat, y_pred_cat, importances = multi_predictor.predict_catboost_multi(X_train_latent, y_train, X_val_latent, y_val)
# print(rmse_cat)#, y_pred_cat)

# Get error on RECONSTRUCTED X (using autoencoder)
# rmse_cat, y_pred_cat, importances = multi_predictor.predict_catboost_multi(X_train_recon, y_train, X_val_recon, y_val)
# print(rmse_cat)#, y_pred_cat)

rmse, y_pred, importances = single_predictor.predict_catboost_single_model(X_train_scaled, y_train_scaled, X_val_scaled, y_val_scaled, cat_features=cat_features)
print(rmse)#, y_pred_cat)

# X_pca_train, X_pca_val, y_train, y_val = train_test_split(X_pca_reduced, y_scaled, test_size=0.2, random_state=42)
# rmse_cat, y_pred_cat, importances = multi_predictor.predict_catboost_multi(X_pca_train, y_train, X_pca_val, y_val)
# print(rmse_cat)#, y_pred_cat)

# Conclusion: compressing to latent, zeroing the least correlated latent col, then decompressing and predicting yields WORSE score

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, root_mean_squared_error

def rmse(y_true, y_pred):
    return np.sqrt(root_mean_squared_error(y_true, y_pred))
rmse_scorer = make_scorer(rmse, greater_is_better=False)

def train_models_in_1_dataset(y_df_expanded, joined_log_spatial_df, main_folder, device):
    """Edited to do everything in 1 dataset"""
    marathon_run_col = "marathon_run"
    wafer_col        = "wafer"
    run_col          = "#run"

    # convert wafer col to str, maybe it helps:
    # joined_log_spatial_df = joined_log_spatial_df.with_columns(pl.col(wafer_col).cast(str))

    preprocessor = PrePredictionProcessor()

    X_train: pd.DataFrame
    y_train: np.ndarray
    X_val:   pd.DataFrame
    y_val:   np.ndarray

    want_to_scale_per_wafer = False
    if want_to_scale_per_wafer:
        X = joined_log_spatial_df.to_pandas().drop(columns=[marathon_run_col, run_col], errors='ignore')
        y = y_df_expanded.drop(marathon_run_col).to_pandas()
        X_train, y_train, X_val, y_val, wafer_x_scalers, wafer_y_scalers = preprocessor.scale_per_wafer_and_split_data(X, y, wafer_col="wafer", test_size=0.2)
    else:
        X = joined_log_spatial_df.to_pandas().drop(columns=[wafer_col, marathon_run_col, run_col], errors='ignore')
        y = y_df_expanded.drop(marathon_run_col, wafer_col).to_pandas()
        # X_train, y_train, X_val, y_val, y_scaler = preprocessor.scale_and_split_data(X, y)
        X_scaled, y_scaled, _ = preprocessor.scale_X_after_split(X, y)
        X_train, X_val, y_train, y_val = train_test_split(X_scaled, y_scaled)

    numeric_cols  = X_train.select_dtypes(include=[np.number]).columns
    zero_var_cols = X_train[numeric_cols].columns[X_train[numeric_cols].var() == 0].tolist()
    X_train_clean = X_train.drop(columns = zero_var_cols)
    X_val_clean   = X_val.drop(columns = zero_var_cols)

    # =-=-=-=-=-= bit about feature importance
    importances = multi_predictor._make_predictions_in_1_function(X_train_clean, y_train, X_val_clean, y_val, device)

    top_features_fraction = 0.05
    X_train_clean_light, X_val_clean_light = LogAndSpatialProcessor.keep_top_features_by_importance(X_train_clean, X_val_clean, importances, top_features_fraction)
    
    # Cross Val score
    model  = RandomForestRegressor()
    scores = cross_val_score(model, X_train_clean_light, y_train, scoring=rmse_scorer, cv=5)
    print(f"CV RMSE mean: {-scores.mean():.3f}")
    
    importances = multi_predictor._make_predictions_in_1_function(X_train_clean_light, y_train, X_val_clean_light, y_val, device)

    # =-=-=-=-=-=-=-=-=-=-=-=

    # ========= bit about correlation
    # correlations = pd.DataFrame({
    #     f"target_{i}": X_train_clean.corrwith(pd.Series(y_train[:, i], index=X_train_clean.index)).abs()
    #     for i in range(y_train.shape[1])})

    # # Average correlations across all targets
    # mean_correlations = correlations.mean(axis=1)

    # # Model importances (make sure index aligns with X_train_clean.columns)
    # importances_mean = pd.Series(mean_importance, index=X_train_clean.columns)

    # # Plot correlation vs importance
    # plt.figure(figsize=(8,6))
    # plt.scatter(mean_correlations, importances_mean)
    # plt.xlabel("Mean Abs(Correlation) with Targets")
    # plt.ylabel("Mean Model Feature Importance")
    # plt.title("Feature Importance vs. Correlation")
    # plt.grid(True)
    # plt.show()
    # ============

    return importances

importances = train_models_in_1_dataset(y_df_expanded, log_df_labelled_no_str, main_folder, device)


In [ ]:

plt.figure(figsize=(17, 8))
# y_values = log_df_labelled_no_str['rc3 signal_1_step4']
y_values = log_df_labelled_no_str['common signal_88_step4']
x_values = range(len(y_values))
plt.scatter(x_values, y_values)
plt.show()


In [ ]:
which_row = 1
y_real_values = (y_df_expanded.row(which_row))[2:]

y_pred_cat_original_scale = y_scaler.inverse_transform(y_pred_cat)
y_val_original_scale = y_scaler.inverse_transform(y_val)
y_pred_values = y_val_original_scale[which_row]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.scatter(range(len(y_real_values)), y_real_values, label='real')
ax.scatter(range(len(y_pred_values)), y_pred_values, label='predicted')
ax.set_xlabel("Site ID")
ax.set_ylabel("Spatial property (nm)")
ax.set_title("Spatial property, real vs predicted")
ax.legend()
plt.show()


In [ ]:
def join_features_targets(big_log_df, target_df):
    # target columns except marathon_run
    target_cols = [str(i) for i in range(1, 110)]
    
    # merge on marathon_run
    full_df = big_log_df.merge(target_df, on='marathon_run', how='left')
    
    # features: drop target columns + marathon_run if not needed as feature
    X = full_df.drop(columns=target_cols + ['marathon_run'])
    
    # targets
    y = full_df[target_cols]
    
    return X, y

# to fix, better to have the object passed onto the function
def train_one_model_for_all_wafers(y_df_dict, radius_wide_dict, big_log_df, marathon_run_col, target_cols, device):
    predictor    = MultiOutputModelPredictor(device)
    preprocessor = PrePredictionProcessor()
    wafer_col    = "wafer"

    # Combine all y dfs into one with wafer column
    y_dfs = []
    for k, df in y_df_dict.items():
        y_dfs.append(df.with_columns(pl.lit(k+1).alias(wafer_col)))
    combined_y_df = pl.concat(y_dfs, how="vertical")

    # Combine all radius dfs into one with wafer column
    radius_dfs = []
    for k, df in radius_wide_dict.items():
        radius_dfs.append(df.with_columns(pl.lit(k+1).alias(wafer_col)))
    combined_radius_df = pl.concat(radius_dfs, how="vertical")

    # Flatten last rows of big_log_df by marathon_run
    processed_log_df = asm._flatten_last_n_rows(big_log_df, marathon_run_col, num_of_last_rows=1)

    # Join features with radius info on marathon_run and wafer
    features_df = processed_log_df.join(
        combined_radius_df,
        on  = [marathon_run_col, wafer_col],
        how = "left")

    # Join features with targets on marathon_run and wafer
    full_df = features_df.join(
        combined_y_df,
        on  = [marathon_run_col, wafer_col],
        how = "inner")

    # Prepare X and y
    y = full_df.select(target_cols).to_pandas()
    X = full_df.drop(target_cols + [marathon_run_col, wafer_col]).to_pandas()

    X = preprocessor.drop_certain_cols_from_df(X, [marathon_run_col])

    # Scale and split
    # X_train, y_train, X_val, y_val, y_scaler = preprocessor.scale_and_split_data(X, y)
    X_scaled, y_scaled, _ = preprocessor.scale_X_after_split(X, y)
    X_train, X_val, y_train, y_val = train_test_split(X_scaled, y_scaled)
    X_train = X_train.fillna(0)
    X_val   = X_val.fillna(0)

    # Drop zero variance cols
    zero_var_cols = X_train.columns[X_train.var() == 0].tolist()
    X_train = X_train.drop(columns=zero_var_cols)
    X_val   = X_val.drop(columns=zero_var_cols)

    # Train all models and print results
    rmse_linreg, _ = predictor.predict_linear_reg(X_train, y_train, X_val, y_val)
    rmse_ridge, _  = predictor.predict_linear_reg_ridge(X_train, y_train, X_val, y_val)
    rmse_lgb, _    = predictor.predict_lightgbm(X_train, y_train, X_val, y_val)
    # rmse_cat, _ = predictor.predict_catboost(X_train, y_train, X_val, y_val)
    rmse_rf, _     = predictor.predict_randomforest(X_train, y_train, X_val, y_val)

    print(f"LinReg RMSE: {rmse_linreg:.3f}")
    print(f"Ridge RMSE: {rmse_ridge:.3f}")
    print(f"LGBM RMSE: {rmse_lgb:.3f}")
    # print(f"Catboost RMSE: {rmse_cat:.3f}")
    print(f"RF RMSE: {rmse_rf:.3f}")

    return {"linreg_rmse": rmse_linreg,
            "ridge_rmse":  rmse_ridge,
            "lgbm_rmse":   rmse_lgb,
            # "catboost_rmse": rmse_cat,
            "rf_rmse":     rmse_rf,}

target_cols = [str(i) for i in range(1, 110)]  # '1' to '109'

results = train_one_model_for_all_wafers(
    y_df_dict        = y_df_dict,
    radius_wide_dict = radius_wide_dict,
    big_log_df       = big_log_df,
    marathon_run_col = "marathon_run",
    target_cols      = target_cols,
    device           = device)

print(results)

# combined_log_df["wafer"]
# print(y_df_dict[0].columns)


In [ ]:

def find_square_periodicity_of_feature(signal):
    """Requires a square function"""
    durations   = np.diff(np.where(np.diff(signal) != 0)[0])
    periods     = durations[::2] + durations[1::2]  # Sum long+short phases
    estimated_p = int(np.round(np.median(periods)))
    return estimated_p


In [ ]:
y_pred_unscaled = y_scaler.inverse_transform(y_pred_cat)

plt.scatter(range(len(y_full_pd.iloc[1].values)), y_full_pd.iloc[1].values,
            label="True", facecolors='none', edgecolors='blue', s=8)
plt.scatter(range(len(y_pred_unscaled[1])), y_pred_unscaled[1],
            label="Predicted", facecolors='none', edgecolors='orange', s=8)
# plt.plot(y_full_pd.iloc[1].values, label="True")
# plt.plot(y_pred_unscaled[1], label="Predicted")
plt.legend()
plt.title("y_predicted vs y_actual")
plt.xlabel("Site ID (coordinate)")
plt.ylabel("Spatial property")
plt.show()


##### Spatial data (M)

In [ ]:
def plot_wafer_property(df, property_col, title):
    plt.figure(figsize=(9, 5))
    for rc_value, group in df.group_by("RC"):
        x = group["#Run"].to_list()
        y = group[property_col].to_list()
        plt.scatter(x, y, label=f'RC {rc_value}', s=20)
    plt.xlabel("#Run")
    plt.ylabel(property_col)
    plt.title(title)
    plt.legend()
    plt.show()

plot_wafer_property(wafer_df, "Wafer property summary 1", "Wafer Property 1")
plot_wafer_property(wafer_df, "Wafer property summary 2", "Wafer Property 2")


##### Import timeseries data (S)

In [ ]:
"""Load data and make parquet files out of it"""

should_we_save_parquet_files = False

def _save_df_as_parquet_file(df: pl.dataframe, saving_location: str):
    df.write_parquet(saving_location)

def remove_unchanging_cols_from_df_and_save(df: pl.dataframe, col_name: str, should_we_save_parquet_files: bool) -> None:
    for run_id in df[col_name].unique().to_list():
        df_per_run    = df.filter(pl.col(col_name) == run_id)
        constant_cols = [col for col in df_per_run.columns
                     if df_per_run[col].dtype in [pl.Int64, pl.Float64] and
                        #  df_per_run.select(pl.col(col).filter(pl.col(col) != 0)).height == 0]
                         df_per_run.select(pl.col(col).n_unique()).item() == 1]
        df_per_run_filtered = df_per_run.drop(constant_cols)
        saving_location = f"{parquet_subfolder}/run_{run_id}.parquet"
        if should_we_save_parquet_files:
            _save_df_as_parquet_file(df_per_run_filtered, saving_location)

remove_unchanging_cols_from_df_and_save(log_df, "#Run", should_we_save_parquet_files)

# =============
# before making funcrtion:

# if should_we_save_parquet_files:
#     for run_id in log_df["#Run"].unique().to_list():
#         df_per_run = log_df.filter(pl.col("#Run") == run_id)
#         zero_cols  = [col for col in df_per_run.columns
#                      if df_per_run[col].dtype in [pl.Int64, pl.Float64] and
#                         #  df_per_run.select(pl.col(col).filter(pl.col(col) != 0)).height == 0]
#                          df_per_run.select(pl.col(col).n_unique()).item() == 1]
#         df_per_run_filtered = df_per_run.drop(zero_cols)
#         df_per_run_filtered.write_parquet(f"{parquet_subfolder}/run_{run_id}.parquet")


##### Timeseries data (S)

In [ ]:
run_number   = 15
parquet_file = f"./ASM_data/3. marathon1/Logs/split_by_run/run_{run_number}.parquet"
df           = pl.read_parquet(parquet_file)
df_pd        = df.to_pandas()
df_numeric   = df.select(pl.col(pl.NUMERIC_DTYPES))
X_np         = df_numeric.to_numpy()
X_scaled     = StandardScaler().fit_transform(X_np)

num_cols_to_plot = len(df_pd.columns)
num_rows         = math.ceil(math.sqrt(num_cols_to_plot))
num_cols_grid    = math.ceil(num_cols_to_plot / num_rows)

axes = df_pd.plot(subplots=True, figsize=(14, 12), layout=(num_rows, num_cols_grid), sharex=True, legend=False)

if isinstance(axes, np.ndarray):
    axes_flat = axes.flatten()
else:
    axes_flat = [axes]

column_names = df_pd.columns.tolist()

for i, ax in enumerate(axes_flat):
    if i < num_cols_to_plot: # Only set title for actual plots
        ax.set_title(column_names[i], fontsize='xx-small')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xticklabels([])
    ax.set_yticklabels([])

for i in range(num_cols_to_plot, len(axes_flat)):
    axes_flat[i].set_visible(False)

plt.tight_layout()
plt.suptitle(f'Run #{run_number} {log_file}', fontsize='large', y=1.02) # Adjust y to prevent overlap
plt.show()